<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/AOC_AGENTIC_GLM5DOT2_DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://z.ai/manage-apikey/subscription

## PART1

In [2]:
import json
import time
import logging
import random
import re
import os
from typing import Dict, List, Any, Optional, Tuple
from datetime import datetime
from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import plotly.graph_objects as go
from collections import defaultdict

# --- 0. Configuration (Logging & API Keys) ---
logging.basicConfig(
    filename='aoc_agent.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Suppress some warnings
import warnings
warnings.filterwarnings('ignore')

# --- Environment Validation ---
def validate_environment():
    """Validate all required environment variables and dependencies."""
    try:
        from google.colab import userdata
        ZAI_KEY_API = userdata.get('ZAI_KEY')
        if not ZAI_KEY_API or ZAI_KEY_API == "PLACEHOLDER_ERROR_KEY":
            raise ValueError("ZAI_KEY not properly configured in Colab Secrets.")
        logger.info("Environment validation: ZAI_KEY found")
        return ZAI_KEY_API
    except ImportError:
        # Not in Colab, try environment variable
        ZAI_KEY_API = os.getenv('ZAI_KEY')
        if not ZAI_KEY_API:
            raise EnvironmentError("ZAI_KEY environment variable not set.")
        logger.info("Environment validation: ZAI_KEY found in environment")
        return ZAI_KEY_API
    except Exception as e:
        logger.error(f"Environment validation failed: {str(e)}")
        raise

# Initialize API Key
try:
    ZAI_KEY_API = validate_environment()
except Exception as e:
    logger.error(f"Fatal: {str(e)}")
    print(f"❌ Fatal Error: {str(e)}")
    print("Please ensure ZAI_KEY is set in Colab Secrets or environment variables.")
    ZAI_KEY_API = None

if not ZAI_KEY_API:
    raise SystemExit("Cannot proceed without valid API key.")

# Initialize Z.ai Client with GLM-5.2 model
client = OpenAI(
    api_key=ZAI_KEY_API,
    base_url="https://api.z.ai/api/paas/v4/"
)

# --- Constants ---
DEFAULT_MODEL = "glm-5.2"
TOOL_TIMEOUT = 30  # seconds
MAX_RETRIES = 3

# --- 1. Tool Registry ---

class ToolRegistry:
    """Registry for all available tools."""

    def __init__(self):
        self._tools = {}
        self._metadata = {}
        self._tool_definitions = []

    def register(self, name: str, func: callable, description: str = "", category: str = "general",
                 parameters: Dict = None, required_params: List[str] = None):
        """Register a tool function with its definition."""
        self._tools[name] = func
        self._metadata[name] = {
            "description": description,
            "category": category,
            "registered_at": datetime.now().isoformat(),
            "parameters": parameters or {},
            "required_params": required_params or []
        }

        # Build tool definition for OpenAI function calling
        if parameters:
            tool_def = {
                "type": "function",
                "function": {
                    "name": name,
                    "description": description,
                    "parameters": {
                        "type": "object",
                        "properties": parameters,
                        "required": required_params or []
                    }
                }
            }
            self._tool_definitions.append(tool_def)

        logger.info(f"Registered tool: {name} ({category})")

    def get(self, name: str) -> Optional[callable]:
        """Get a tool by name."""
        return self._tools.get(name)

    def list_tools(self) -> List[str]:
        """List all registered tool names."""
        return list(self._tools.keys())

    def get_metadata(self, name: str) -> Dict:
        """Get metadata for a tool."""
        return self._metadata.get(name, {})

    def get_tools_by_category(self, category: str) -> List[str]:
        """Get tools by category."""
        return [name for name, meta in self._metadata.items()
                if meta.get("category") == category]

    def get_tool_definitions(self) -> List[Dict]:
        """Get OpenAI function calling definitions for all tools."""
        return self._tool_definitions

    def get_tool_definition(self, name: str) -> Optional[Dict]:
        """Get OpenAI function calling definition for a specific tool."""
        for tool_def in self._tool_definitions:
            if tool_def["function"]["name"] == name:
                return tool_def
        return None

# Initialize global tool registry
tool_registry = ToolRegistry()

# --- 2. Core Tool Definitions ---

def validate_json_string(data: str) -> bool:
    """Validate if string is proper JSON."""
    try:
        json.loads(data)
        return True
    except (json.JSONDecodeError, TypeError):
        return False

def safe_json_loads(data: str, default: Any = None) -> Any:
    """Safely load JSON with fallback."""
    if not data:
        return default or {}
    try:
        return json.loads(data)
    except json.JSONDecodeError as e:
        logger.warning(f"JSON parsing failed: {e}")
        return default or {}

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10),
       retry=retry_if_exception_type(Exception))
def get_flight_status(flight_number: str, departure_icao: str = None) -> str:
    """
    Retrieve real-time flight status by flight number.

    Args:
        flight_number: Flight number, e.g., AA123
        departure_icao: ICAO code of departure airport, optional

    Returns:
        JSON string with flight details
    """
    logger.info(f"Fetching flight status for {flight_number}")

    if not flight_number or not isinstance(flight_number, str):
        raise ValueError(f"Invalid flight_number: {flight_number}")

    mock_data = {
        "AA123": {
            "flight_number": "AA123",
            "status": "delayed",
            "delay_minutes": 45,
            "eta": "2025-10-01T18:30:00Z",
            "gate": "B12",
            "aircraft": "B737",
            "departure_icao": "KJFK",
            "destination_icao": "KLAX"
        },
        "UA456": {
            "flight_number": "UA456",
            "status": "on_time",
            "delay_minutes": 0,
            "eta": "2025-10-01T17:45:00Z",
            "gate": "C5",
            "aircraft": "A320",
            "departure_icao": "KORD",
            "destination_icao": "KSFO"
        },
        "DL789": {
            "flight_number": "DL789",
            "status": "delayed",
            "delay_minutes": 60,
            "eta": "2025-10-01T19:00:00Z",
            "gate": "A10",
            "aircraft": "A321",
            "departure_icao": "KJFK",
            "destination_icao": "KORD"
        }
    }

    data = mock_data.get(flight_number, {
        "flight_number": flight_number,
        "status": "unknown",
        "delay_minutes": 0,
        "eta": "Unknown",
        "gate": "Unknown",
        "aircraft": "Unknown",
        "departure_icao": departure_icao or "Unknown",
        "destination_icao": "Unknown"
    })

    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def get_aviation_weather(icao_code: str) -> str:
    """
    Fetch weather data for an airport by ICAO code.

    Args:
        icao_code: ICAO code, e.g., KJFK

    Returns:
        JSON string with weather data
    """
    logger.info(f"Fetching weather for {icao_code}")

    if not isinstance(icao_code, str) or not icao_code:
        logger.warning(f"Invalid icao_code: {icao_code}, defaulting to KJFK")
        icao_code = "KJFK"

    mock_data = {
        "KJFK": {
            "icao": "KJFK",
            "metar": "KJFK 011730Z 27010KT 10SM FEW030 BKN050 15/10 A2992",
            "conditions": "Light wind, clear skies",
            "temperature": 15,
            "wind_speed": 10,
            "visibility": 10
        },
        "KLAX": {
            "icao": "KLAX",
            "metar": "KLAX 011745Z 18008KT 5SM HZ BKN040 22/18 A2985",
            "conditions": "Haze, moderate visibility",
            "temperature": 22,
            "wind_speed": 8,
            "visibility": 5
        },
        "KORD": {
            "icao": "KORD",
            "metar": "KORD 011800Z 30012KT 8SM OVC035 12/08 A2990",
            "conditions": "Overcast, windy",
            "temperature": 12,
            "wind_speed": 12,
            "visibility": 8
        }
    }

    data = mock_data.get(icao_code, {
        "icao": icao_code,
        "metar": f"{icao_code} 011730Z 27010KT 10SM CLR 20/15 A2995",
        "conditions": "Assumed clear conditions (fallback)",
        "temperature": 20,
        "wind_speed": 10,
        "visibility": 10
    })
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def check_crew_availability(flight_number: str) -> str:
    """
    Check crew availability for a flight.

    Args:
        flight_number: Flight number, e.g., AA123

    Returns:
        JSON string with crew availability details
    """
    logger.info(f"Checking crew availability for {flight_number}")

    mock_data = {
        "AA123": {
            "available": True,
            "pilot": "Available (ID: P001, Rest: 10h)",
            "copilot": "Available (ID: C001, Rest: 9h)",
            "cabin_crew": ["Available (ID: CC001, Rest: 8h)", "Available (ID: CC002, Rest: 8h)"],
            "total_crew": 4,
            "rest_compliance": True
        },
        "UA456": {
            "available": False,
            "pilot": "Unavailable (ID: P002, Rest: 2h)",
            "copilot": "Available (ID: C002, Rest: 7h)",
            "cabin_crew": ["Available (ID: CC003, Rest: 6h)"],
            "total_crew": 2,
            "rest_compliance": False
        }
    }

    data = mock_data.get(flight_number, {
        "available": False,
        "pilot": "Unknown",
        "copilot": "Unknown",
        "cabin_crew": [],
        "total_crew": 0,
        "rest_compliance": False
    })
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def estimate_passenger_impact(flight_number: str, delay_minutes: int) -> str:
    """
    Estimate passenger impact due to delay.

    Args:
        flight_number: Flight number
        delay_minutes: Delay duration in minutes

    Returns:
        JSON string with passenger impact estimates
    """
    logger.info(f"Estimating passenger impact for {flight_number}, delay: {delay_minutes}")

    passengers = random.randint(100, 200)
    rebooking_cost = 200 * passengers if delay_minutes > 60 else 0
    compensation_cost = 50 * passengers if delay_minutes > 120 else 0
    meal_vouchers = 15 * passengers if delay_minutes > 90 else 0
    hotel_cost = 150 * passengers if delay_minutes > 180 else 0

    total_cost = rebooking_cost + compensation_cost + meal_vouchers + hotel_cost

    data = {
        "flight_number": flight_number,
        "passengers": passengers,
        "rebooking_cost": rebooking_cost,
        "compensation_cost": compensation_cost,
        "meal_vouchers": meal_vouchers,
        "hotel_cost": hotel_cost,
        "total_cost": total_cost
    }
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def predict_delay_escalation(flight_number: str, delay_minutes: int) -> str:
    """
    Predict likelihood of delay escalation.

    Args:
        flight_number: Flight number
        delay_minutes: Current delay in minutes

    Returns:
        JSON string with escalation predictions
    """
    logger.info(f"Predicting delay escalation for {flight_number}, delay: {delay_minutes}")

    # More sophisticated prediction model
    base_likelihood = min(0.1 + delay_minutes * 0.005, 0.9)

    # Consider time of day factor (simulated)
    hour = datetime.now().hour
    time_factor = 1.0
    if 6 <= hour < 9 or 16 <= hour < 19:  # Peak hours
        time_factor = 1.3
    elif 22 <= hour or hour < 5:  # Night
        time_factor = 0.7

    likelihood = min(base_likelihood * time_factor, 0.95)

    # Predicted additional delay
    additional_delay = int(max(15, delay_minutes * 0.5 * (1 + likelihood)))

    data = {
        "escalation_likelihood": round(likelihood, 3),
        "predicted_additional_delay": additional_delay,
        "time_factor": time_factor,
        "confidence": 0.85 - (delay_minutes / 1000)  # Confidence decreases with delay
    }
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def check_regulatory_compliance(flight_data: str, crew_data: str) -> str:
    """
    Check compliance with aviation regulations.

    Args:
        flight_data: JSON string of flight data
        crew_data: JSON string of crew data

    Returns:
        JSON string with compliance status
    """
    logger.info("Checking regulatory compliance")

    try:
        flight = safe_json_loads(flight_data)
        crew = safe_json_loads(crew_data)

        issues = []
        warnings = []

        # Check crew compliance
        if not crew.get("available", False):
            issues.append("Crew rest non-compliant")
        if not crew.get("rest_compliance", False):
            issues.append("Crew rest period insufficient")

        # Check delay compliance
        delay = flight.get("delay_minutes", 0)
        if delay > 180:
            issues.append("Delay exceeds regulatory threshold (180 min)")
        elif delay > 120:
            warnings.append("Delay approaching regulatory threshold (180 min)")

        # Check crew count
        if crew.get("total_crew", 0) < 3:
            warnings.append("Minimum crew complement may be insufficient")

        data = {
            "status": "Non-compliant" if issues else "Compliant with warnings" if warnings else "Fully Compliant",
            "issues": issues,
            "warnings": warnings,
            "compliance_score": 1.0 - (len(issues) * 0.3) - (len(warnings) * 0.1)
        }
        return json.dumps(data)

    except Exception as e:
        logger.error(f"Error in check_regulatory_compliance: {str(e)}")
        return json.dumps({"error": str(e), "status": "Unknown"})

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def estimate_passenger_feedback(flight_number: str, delay_minutes: int) -> str:
    """
    Estimate passenger feedback based on delay.

    Args:
        flight_number: Flight number
        delay_minutes: Delay duration in minutes

    Returns:
        JSON string with satisfaction estimates
    """
    logger.info(f"Estimating passenger feedback for {flight_number}, delay: {delay_minutes}")

    # Base satisfaction decreases with delay
    satisfaction = max(0.1, 0.9 - delay_minutes * 0.005)

    # Add some randomness
    satisfaction = min(0.99, satisfaction + random.uniform(-0.05, 0.05))

    # Determine feedback category
    if satisfaction > 0.75:
        feedback = "Positive"
        sentiment_score = 0.8
    elif satisfaction > 0.5:
        feedback = "Neutral"
        sentiment_score = 0.5
    elif satisfaction > 0.3:
        feedback = "Negative"
        sentiment_score = 0.2
    else:
        feedback = "Very Negative"
        sentiment_score = 0.05

    data = {
        "flight_number": flight_number,
        "satisfaction_score": round(satisfaction, 3),
        "feedback": feedback,
        "sentiment_score": sentiment_score,
        "delay_impact": "High" if delay_minutes > 120 else "Medium" if delay_minutes > 60 else "Low"
    }
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def get_atc_update(flight_number: str) -> str:
    """
    Fetch ATC updates for a flight.

    Args:
        flight_number: Flight number

    Returns:
        JSON string with ATC information
    """
    logger.info(f"Fetching ATC update for {flight_number}")

    # Simulated ATC responses
    atc_responses = [
        {"status": "Clearance expected in 30 min", "additional_delay_min": 30, "confidence": "High"},
        {"status": "Holding pattern expected", "additional_delay_min": 45, "confidence": "Medium"},
        {"status": "Immediate clearance available", "additional_delay_min": 0, "confidence": "High"},
        {"status": "Weather delay at destination", "additional_delay_min": 60, "confidence": "Low"}
    ]

    data = random.choice(atc_responses)
    data["flight_number"] = flight_number
    data["timestamp"] = datetime.now().isoformat()

    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def calculate_cost_threshold(flight_data: str, passenger_data: str, prediction_data: str) -> str:
    """
    Calculate cost thresholds for delay vs. cancellation.

    Args:
        flight_data: JSON string of flight data
        passenger_data: JSON string of passenger data
        prediction_data: JSON string of prediction data

    Returns:
        JSON string with cost analysis
    """
    logger.info("Calculating cost threshold")

    try:
        flight = safe_json_loads(flight_data)
        passenger = safe_json_loads(passenger_data)
        prediction = safe_json_loads(prediction_data)

        delay_min = flight.get("delay_minutes", 0)
        passengers = passenger.get("passengers", 100)

        # Cost calculations
        fixed_operational_cost = 25000
        fuel_cost_per_min = 150
        crew_cost_per_hour = 500

        # Delay costs
        delay_cost = (delay_min * fuel_cost_per_min) + \
                    (delay_min / 60 * crew_cost_per_hour * 3) + \
                    passenger.get("total_cost", 0)

        # Cancellation costs
        cancel_cost = 50000 + (passengers * 350)  # Rebooking + compensation

        # Additional costs for predicted escalation
        escalation_extra = prediction.get("predicted_additional_delay", 0) * 150
        delay_cost_with_escalation = delay_cost + escalation_extra

        # Revenue loss
        avg_ticket_price = 450
        revenue_loss = passengers * avg_ticket_price * 0.3  # 30% may not rebook

        total_cancel_cost = cancel_cost + revenue_loss

        data = {
            "delay_cost": int(delay_cost),
            "delay_cost_with_escalation": int(delay_cost_with_escalation),
            "cancel_cost": int(total_cancel_cost),
            "cost_difference": int(total_cancel_cost - delay_cost),
            "threshold_min": 120,  # minutes
            "current_delay": delay_min,
            "recommendation": "Cancel" if delay_min > 120 or (delay_min > 60 and prediction.get("escalation_likelihood", 0) > 0.7) else "Delay",
            "confidence": min(0.95, 0.8 + (1 - prediction.get("escalation_likelihood", 0)))
        }
        return json.dumps(data)

    except Exception as e:
        logger.error(f"Error in calculate_cost_threshold: {str(e)}")
        return json.dumps({"error": str(e)})

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def pre_plan_crew_swap(flight_number: str, crew_data: str, delay_minutes: int) -> str:
    """
    Pre-plan crew swap if needed.

    Args:
        flight_number: Flight number
        crew_data: JSON string of crew data
        delay_minutes: Delay duration in minutes

    Returns:
        JSON string with crew swap recommendations
    """
    logger.info(f"Pre-planning crew swap for {flight_number}, delay: {delay_minutes}")

    try:
        crew = safe_json_loads(crew_data)

        # Determine if swap is needed
        need_swap = not crew.get("available", False)

        # If delay > 120 min, may need swap even if available (rest period ending)
        if delay_minutes > 120 and crew.get("available", False):
            # Check rest periods
            pilot_rest = crew.get("pilot", "")
            if "Rest: " in pilot_rest:
                rest_hours = int(re.search(r'Rest: (\d+)h', pilot_rest).group(1)) if re.search(r'Rest: (\d+)h', pilot_rest) else 10
                if rest_hours < 4:
                    need_swap = True

        recommendation = "Swap required" if need_swap else "No swap needed"

        data = {
            "recommendation": recommendation,
            "need_swap": need_swap,
            "standby_crew": None if not need_swap else {
                "pilot": "P002 (Available, 12h rest)",
                "copilot": "C002 (Available, 11h rest)",
                "cabin_crew": ["CC003 (Available, 10h rest)", "CC004 (Available, 9h rest)"]
            },
            "swap_duration_min": 30 if need_swap else 0,
            "swap_cost": 2000 if need_swap else 0
        }
        return json.dumps(data)

    except Exception as e:
        logger.error(f"Error in pre_plan_crew_swap: {str(e)}")
        return json.dumps({"error": str(e), "recommendation": "Unable to assess"})

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def get_alternate_routes(flight_number: str, departure_icao: str, destination_icao: str) -> str:
    """
    Fetch alternate routes for a flight.

    Args:
        flight_number: Flight number
        departure_icao: ICAO code of departure
        destination_icao: ICAO code of destination

    Returns:
        JSON string with route alternatives
    """
    logger.info(f"Fetching alternate routes for {flight_number} from {departure_icao} to {destination_icao}")

    # Common alternate airports
    alternates = {
        "KJFK-KLAX": [
            {"route": "KJFK-KPHL-KLAX", "extra_time_min": 20, "extra_cost": 3000, "extra_fuel": 1500, "risk": "low"},
            {"route": "KJFK-KORD-KLAX", "extra_time_min": 45, "extra_cost": 5000, "extra_fuel": 3000, "risk": "medium"},
            {"route": "KJFK-KDFW-KLAX", "extra_time_min": 60, "extra_cost": 6500, "extra_fuel": 4000, "risk": "medium"}
        ],
        "KORD-KSFO": [
            {"route": "KORD-KDEN-KSFO", "extra_time_min": 25, "extra_cost": 3500, "extra_fuel": 2000, "risk": "low"},
            {"route": "KORD-KSLC-KSFO", "extra_time_min": 40, "extra_cost": 4500, "extra_fuel": 2800, "risk": "medium"}
        ]
    }

    route_key = f"{departure_icao}-{destination_icao}"
    alt_routes = alternates.get(route_key, [
        {"route": f"{departure_icao}-KATL-{destination_icao}", "extra_time_min": 30, "extra_cost": 4000, "extra_fuel": 2500, "risk": "low"},
        {"route": f"{departure_icao}-KDEN-{destination_icao}", "extra_time_min": 50, "extra_cost": 5500, "extra_fuel": 3500, "risk": "medium"}
    ])

    data = {
        "original_route": f"{departure_icao}-{destination_icao} direct",
        "alternates": alt_routes,
        "weather_impacts": {
            "KPHL": "Clear",
            "KORD": "Light rain",
            "KDEN": "Snow possible"
        }
    }
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def update_passenger_feedback(flight_number: str, satisfaction_score: float, action: str) -> str:
    """
    Update passenger feedback based on action taken.

    Args:
        flight_number: Flight number
        satisfaction_score: Current satisfaction score
        action: Action taken, e.g., provide amenities

    Returns:
        JSON string with updated feedback
    """
    logger.info(f"Updating passenger feedback for {flight_number}, action: {action}")

    # Determine impact of action
    positive_actions = ["amenities", "voucher", "upgrade", "compensation", "communication"]
    negative_actions = ["delay", "cancel", "deny"]

    impact = 0
    action_lower = action.lower()

    for pos in positive_actions:
        if pos in action_lower:
            impact += 0.1
    for neg in negative_actions:
        if neg in action_lower:
            impact -= 0.2

    # Apply impact
    new_score = min(0.95, max(0.05, satisfaction_score + impact))

    # Add some randomness
    new_score = min(0.95, max(0.05, new_score + random.uniform(-0.02, 0.02)))

    # Determine feedback
    if new_score > 0.75:
        feedback = "Positive"
    elif new_score > 0.5:
        feedback = "Neutral"
    elif new_score > 0.25:
        feedback = "Negative"
    else:
        feedback = "Very Negative"

    data = {
        "flight_number": flight_number,
        "previous_satisfaction": round(satisfaction_score, 3),
        "satisfaction_score": round(new_score, 3),
        "feedback": feedback,
        "action_taken": action,
        "improvement": round(new_score - satisfaction_score, 3),
        "timestamp": datetime.now().isoformat()
    }
    return json.dumps(data)

# --- 3. Build AI Decision Function with Tool Definitions ---

def build_decision_context(parsed_data: Dict[str, Any]) -> str:
    """
    Build a comprehensive context string for the AI model.
    """
    flight = parsed_data.get('flight', {})
    weather = parsed_data.get('weather', {})
    crew = parsed_data.get('crew', {})
    routes = parsed_data.get('routes', {})
    passenger = parsed_data.get('passenger', {})
    prediction = parsed_data.get('prediction', {})
    compliance = parsed_data.get('compliance', {})
    feedback = parsed_data.get('feedback', {})
    atc = parsed_data.get('atc', {})
    threshold = parsed_data.get('threshold', {})

    delay_min = flight.get('delay_minutes', 0)
    flight_number = flight.get('flight_number', 'Unknown')

    context = f"""
    FLIGHT OPERATIONS CONTEXT
    =========================

    FLIGHT INFORMATION:
    • Flight Number: {flight_number}
    • Status: {flight.get('status', 'unknown').upper()}
    • Current Delay: {delay_min} minutes
    • Gate: {flight.get('gate', 'Unknown')}
    • Aircraft: {flight.get('aircraft', 'Unknown')}
    • Route: {flight.get('departure_icao', 'Unknown')} → {flight.get('destination_icao', 'Unknown')}
    • ETA: {flight.get('eta', 'Unknown')}

    WEATHER CONDITIONS:
    • Airport: {weather.get('icao', 'Unknown')}
    • Conditions: {weather.get('conditions', 'Unknown')}
    • Temperature: {weather.get('temperature', 'N/A')}°C
    • Wind Speed: {weather.get('wind_speed', 'N/A')} knots
    • Visibility: {weather.get('visibility', 'N/A')} miles
    • METAR: {weather.get('metar', 'Unknown')}

    CREW STATUS:
    • Available: {'YES' if crew.get('available', False) else 'NO'}
    • Pilot: {crew.get('pilot', 'Unknown')}
    • Copilot: {crew.get('copilot', 'Unknown')}
    • Total Crew: {crew.get('total_crew', 0)}
    • Rest Compliance: {'YES' if crew.get('rest_compliance', False) else 'NO'}

    PASSENGER IMPACT:
    • Passengers: {passenger.get('passengers', 0)}
    • Rebooking Cost: ${passenger.get('rebooking_cost', 0):,}
    • Meal Vouchers: ${passenger.get('meal_vouchers', 0):,}
    • Total Passenger Cost: ${passenger.get('total_cost', 0):,}

    PREDICTIONS:
    • Escalation Likelihood: {prediction.get('escalation_likelihood', 0) * 100:.1f}%
    • Predicted Additional Delay: {prediction.get('predicted_additional_delay', 0)} minutes
    • Prediction Confidence: {prediction.get('confidence', 0) * 100:.1f}%

    REGULATORY COMPLIANCE:
    • Status: {compliance.get('status', 'Unknown')}
    • Compliance Score: {compliance.get('compliance_score', 0) * 100:.1f}%
    • Issues: {', '.join(compliance.get('issues', [])) or 'None'}
    • Warnings: {', '.join(compliance.get('warnings', [])) or 'None'}

    PASSENGER FEEDBACK:
    • Satisfaction Score: {feedback.get('satisfaction_score', 0):.2f}
    • Sentiment: {feedback.get('feedback', 'Unknown')}
    • Delay Impact: {feedback.get('delay_impact', 'Unknown')}

    ATC UPDATE:
    • Status: {atc.get('status', 'Unknown')}
    • Additional Delay: {atc.get('additional_delay_min', 0)} minutes
    • Confidence: {atc.get('confidence', 'Unknown')}

    COST ANALYSIS:
    • Delay Cost (Current): ${threshold.get('delay_cost', 0):,}
    • Delay Cost with Escalation: ${threshold.get('delay_cost_with_escalation', 0):,}
    • Cancellation Cost: ${threshold.get('cancel_cost', 0):,}
    • Cost Difference: ${threshold.get('cost_difference', 0):,}
    • Cost Recommendation: {threshold.get('recommendation', 'Unknown')}

    ROUTE ALTERNATIVES:
    • Original: {routes.get('original_route', 'Unknown')}
    • Alternatives: {len(routes.get('alternates', []))} available
    """

    # Add route details
    for i, alt in enumerate(routes.get('alternates', []), 1):
        context += f"\n    {i}. {alt.get('route', 'Unknown')} (+{alt.get('extra_time_min', 0)} min, ${alt.get('extra_cost', 0):,})"

    return context

def extract_json_from_response(response: str) -> Optional[Dict]:
    """
    Extract JSON from AI response with multiple strategies.
    """
    # Strategy 1: Try to parse the entire response as JSON
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        pass

    # Strategy 2: Look for JSON in markdown code blocks
    code_block_pattern = r'```(?:json)?\s*(\{.*?\})\s*```'
    matches = re.findall(code_block_pattern, response, re.DOTALL)
    for match in matches:
        try:
            return json.loads(match)
        except json.JSONDecodeError:
            continue

    # Strategy 3: Find JSON-like structures with regex
    json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    matches = re.findall(json_pattern, response, re.DOTALL)

    # Try the longest match first (most likely to be the complete JSON)
    matches.sort(key=len, reverse=True)
    for match in matches:
        try:
            # Clean up the JSON string
            cleaned = match.strip()
            # Fix common JSON issues
            cleaned = re.sub(r',\s*}', '}', cleaned)
            cleaned = re.sub(r',\s*]', ']', cleaned)
            return json.loads(cleaned)
        except json.JSONDecodeError:
            continue

    return None

def suggest_operations_decision_ai(
    flight_data: str, weather_data: str, crew_data: str, route_data: str,
    passenger_data: str, prediction_data: str, compliance_data: str,
    feedback_data: str, atc_data: str, threshold_data: str, priority: str,
    model: str = DEFAULT_MODEL) -> str:
    """
    Use GLM-5.2 AI model with function calling to generate intelligent operations decisions.
    """
    logger.info(f"Generating AI-powered decision with {model}, priority: {priority}")

    try:
        # Parse all input data with validation
        parsed_data = {}
        data_map = {
            'flight': flight_data,
            'weather': weather_data,
            'crew': crew_data,
            'routes': route_data,
            'passenger': passenger_data,
            'prediction': prediction_data,
            'compliance': compliance_data,
            'feedback': feedback_data,
            'atc': atc_data,
            'threshold': threshold_data
        }

        for name, data_str in data_map.items():
            parsed = safe_json_loads(data_str, {})
            if parsed:
                parsed_data[name] = parsed
            else:
                logger.warning(f"Failed to parse {name} data, using defaults")
                parsed_data[name] = {}

        # Build context
        context = build_decision_context(parsed_data)

        flight_number = parsed_data.get('flight', {}).get('flight_number', 'Unknown')
        delay_min = parsed_data.get('flight', {}).get('delay_minutes', 0)

        # System prompt for GLM-5.2
        system_prompt = """You are a senior aviation operations manager with over 20 years of experience in airline operations,
        flight dispatch, crisis management, and passenger services. You excel at making complex operational decisions that
        balance safety, financial performance, passenger satisfaction, and regulatory compliance."""

        # User prompt
        user_prompt = f"""
        Based on the flight operations data provided, generate a decision recommendation prioritizing {priority.upper()}.

        DATA:
        {context}

        Provide your recommendation as a JSON object with these exact fields:
        {{
            "primary_recommendation": "Clear action to take",
            "secondary_actions": ["Action 1", "Action 2", "Action 3"],
            "risk_assessment": {{
                "level": "Low|Medium-Low|Medium|Medium-High|High",
                "rationale": "Why this risk level"
            }},
            "estimated_cost": "Specific cost estimate",
            "passenger_impact": {{
                "current_impact": "Current passenger situation",
                "expected_impact": "Expected after actions",
                "mitigation": "How to help passengers"
            }},
            "timeline": {{
                "immediate_actions": "0-30 min",
                "short_term": "30-120 min",
                "medium_term": "2-6 hours"
            }},
            "justification": "Detailed reasoning for this recommendation"
        }}

        Return ONLY valid JSON, no other text.
        """

        # Get tool definitions for function calling
        tools = tool_registry.get_tool_definitions()

        # Call GLM-5.2 API with function calling
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                tools=tools,
                tool_choice="auto",
                temperature=0.2,
                max_tokens=2000,
                top_p=0.85
            )

            response_message = completion.choices[0].message
            ai_response = response_message.content or ""

            # Check if the model wants to call a function
            if response_message.tool_calls:
                logger.info(f"Model requested {len(response_message.tool_calls)} tool calls")

                # Execute the tool calls
                tool_results = []
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    # Get the function
                    func = tool_registry.get(function_name)
                    if func:
                        try:
                            # Execute the function
                            result = func(**function_args)
                            tool_results.append({
                                "tool": function_name,
                                "result": safe_json_loads(result, {})
                            })
                            logger.info(f"Executed tool: {function_name}")
                        except Exception as e:
                            logger.error(f"Error executing tool {function_name}: {e}")
                            tool_results.append({
                                "tool": function_name,
                                "error": str(e)
                            })
                    else:
                        logger.warning(f"Tool {function_name} not found")

                # Include tool results in the response
                if tool_results:
                    ai_response += f"\n\nTool execution results: {json.dumps(tool_results, indent=2)}"

            logger.info(f"AI Response received (length: {len(ai_response)} chars)")

        except Exception as api_error:
            logger.error(f"API call failed: {str(api_error)}")
            # Fallback without tools
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.2,
                max_tokens=2000
            )
            ai_response = completion.choices[0].message.content

        # Extract JSON from response
        decision_data = extract_json_from_response(ai_response)

        if not decision_data:
            logger.warning("Could not extract JSON from AI response, using fallback")
            decision_data = {
                "primary_recommendation": "AI analysis completed but JSON parsing failed",
                "secondary_actions": ["Please check the full response for details"],
                "risk_assessment": {"level": "Medium", "rationale": "Unable to parse full response"},
                "estimated_cost": "$0",
                "passenger_impact": {
                    "current_impact": "See full response",
                    "expected_impact": "See full response",
                    "mitigation": "See full response"
                },
                "timeline": {
                    "immediate_actions": "See full response",
                    "short_term": "See full response",
                    "medium_term": "See full response"
                },
                "justification": ai_response[:500] + ("..." if len(ai_response) > 500 else ""),
                "raw_response": ai_response
            }

        # Ensure required fields exist
        required_fields = ['primary_recommendation', 'secondary_actions', 'risk_assessment',
                         'estimated_cost', 'passenger_impact', 'timeline', 'justification']

        for field in required_fields:
            if field not in decision_data:
                if field == 'secondary_actions':
                    decision_data[field] = []
                elif field == 'risk_assessment':
                    decision_data[field] = {"level": "Medium", "rationale": "Not specified"}
                elif field == 'passenger_impact':
                    decision_data[field] = {"current_impact": "Not specified", "expected_impact": "Not specified", "mitigation": "Not specified"}
                elif field == 'timeline':
                    decision_data[field] = {"immediate_actions": "Not specified", "short_term": "Not specified", "medium_term": "Not specified"}
                else:
                    decision_data[field] = "Not specified"

        # Add metadata
        decision_data['model_used'] = model
        decision_data['priority'] = priority
        decision_data['flight_number'] = flight_number
        decision_data['timestamp'] = datetime.now().isoformat()
        decision_data['delay_minutes'] = delay_min

        return json.dumps(decision_data, indent=2)

    except Exception as e:
        logger.error(f"Critical error in AI decision making: {str(e)}", exc_info=True)
        return json.dumps({
            "error": f"AI decision failed: {str(e)}",
            "primary_recommendation": "Proceed with standard delay procedures",
            "secondary_actions": ["Monitor situation", "Communicate with passengers", "Coordinate with ATC"],
            "risk_assessment": {"level": "Medium", "rationale": "Fallback decision due to AI error"},
            "estimated_cost": "$0",
            "passenger_impact": {"current_impact": "Under evaluation", "expected_impact": "To be determined", "mitigation": "To be determined"},
            "timeline": {"immediate_actions": "Begin standard procedures", "short_term": "Monitor and update", "medium_term": "Re-evaluate"},
            "justification": "Generated as fallback due to AI processing error",
            "timestamp": datetime.now().isoformat(),
            "priority": priority,
            "fallback": True
        }, indent=2)

# --- 4. Register All Tools with Definitions ---

# Register each tool with its parameter definitions
tool_registry.register(
    "get_flight_status", get_flight_status,
    description="Retrieve real-time flight status by flight number.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number, e.g., AA123"},
        "departure_icao": {"type": "string", "description": "ICAO code of departure airport, optional"}
    },
    required_params=["flight_number"],
    category="flight_operations"
)

tool_registry.register(
    "get_aviation_weather", get_aviation_weather,
    description="Fetch weather data for an airport by ICAO code.",
    parameters={
        "icao_code": {"type": "string", "description": "ICAO code, e.g., KJFK"}
    },
    required_params=["icao_code"],
    category="weather"
)

tool_registry.register(
    "check_crew_availability", check_crew_availability,
    description="Check crew availability for a flight.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number, e.g., AA123"}
    },
    required_params=["flight_number"],
    category="crew_management"
)

tool_registry.register(
    "estimate_passenger_impact", estimate_passenger_impact,
    description="Estimate passenger impact due to delay.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "delay_minutes": {"type": "integer", "description": "Delay duration in minutes"}
    },
    required_params=["flight_number", "delay_minutes"],
    category="passenger_services"
)

tool_registry.register(
    "predict_delay_escalation", predict_delay_escalation,
    description="Predict likelihood of delay escalation.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "delay_minutes": {"type": "integer", "description": "Current delay in minutes"}
    },
    required_params=["flight_number", "delay_minutes"],
    category="predictions"
)

tool_registry.register(
    "check_regulatory_compliance", check_regulatory_compliance,
    description="Check compliance with aviation regulations.",
    parameters={
        "flight_data": {"type": "string", "description": "JSON string of flight data"},
        "crew_data": {"type": "string", "description": "JSON string of crew data"}
    },
    required_params=["flight_data", "crew_data"],
    category="compliance"
)

tool_registry.register(
    "estimate_passenger_feedback", estimate_passenger_feedback,
    description="Estimate passenger feedback based on delay.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "delay_minutes": {"type": "integer", "description": "Delay duration in minutes"}
    },
    required_params=["flight_number", "delay_minutes"],
    category="passenger_services"
)

tool_registry.register(
    "get_atc_update", get_atc_update,
    description="Fetch ATC updates for a flight.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"}
    },
    required_params=["flight_number"],
    category="flight_operations"
)

tool_registry.register(
    "calculate_cost_threshold", calculate_cost_threshold,
    description="Calculate cost thresholds for delay vs. cancellation.",
    parameters={
        "flight_data": {"type": "string", "description": "JSON string of flight data"},
        "passenger_data": {"type": "string", "description": "JSON string of passenger data"},
        "prediction_data": {"type": "string", "description": "JSON string of prediction data"}
    },
    required_params=["flight_data", "passenger_data", "prediction_data"],
    category="cost_analysis"
)

tool_registry.register(
    "pre_plan_crew_swap", pre_plan_crew_swap,
    description="Pre-plan crew swap if needed.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "crew_data": {"type": "string", "description": "JSON string of crew data"},
        "delay_minutes": {"type": "integer", "description": "Delay duration in minutes"}
    },
    required_params=["flight_number", "crew_data", "delay_minutes"],
    category="crew_management"
)

tool_registry.register(
    "get_alternate_routes", get_alternate_routes,
    description="Fetch alternate routes for a flight.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "departure_icao": {"type": "string", "description": "ICAO code of departure"},
        "destination_icao": {"type": "string", "description": "ICAO code of destination"}
    },
    required_params=["flight_number", "departure_icao", "destination_icao"],
    category="flight_operations"
)

tool_registry.register(
    "update_passenger_feedback", update_passenger_feedback,
    description="Update passenger feedback based on action taken.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "satisfaction_score": {"type": "number", "description": "Current satisfaction score"},
        "action": {"type": "string", "description": "Action taken, e.g., provide amenities"}
    },
    required_params=["flight_number", "satisfaction_score", "action"],
    category="passenger_services"
)

# --- 5. Legacy Decision Function ---

def suggest_operations_decision_legacy(
    flight_data: str, weather_data: str, crew_data: str, route_data: str,
    passenger_data: str, prediction_data: str, compliance_data: str,
    feedback_data: str, atc_data: str, threshold_data: str, priority: str
) -> str:
    """Legacy rule-based decision function."""
    logger.info(f"Using legacy decision with priority: {priority}")

    try:
        flight = safe_json_loads(flight_data)
        weather = safe_json_loads(weather_data)
        crew = safe_json_loads(crew_data)
        routes = safe_json_loads(route_data)
        passenger = safe_json_loads(passenger_data)
        prediction = safe_json_loads(prediction_data)
        compliance = safe_json_loads(compliance_data)
        feedback = safe_json_loads(feedback_data)
        atc = safe_json_loads(atc_data)
        threshold = safe_json_loads(threshold_data)

        delay_min = flight.get("delay_minutes", 0)

        # Early return for no delay
        if delay_min == 0:
            return json.dumps({
                "recommendation": [{"action": "Proceed on schedule", "priority": 1, "risk": "low"}],
                "estimated_cost": "$0",
                "risk_level": "low",
                "type": "legacy"
            })

        # Build recommendations based on rules
        recommendation = []
        total_cost = 0

        # Determine risk level
        escalation_likelihood = prediction.get("escalation_likelihood", 0.5)
        risk_level = "high" if escalation_likelihood > 0.6 else "medium" if escalation_likelihood > 0.3 else "low"

        # Check compliance
        if compliance.get("status") != "Fully Compliant":
            recommendation.append({
                "action": "Address compliance issues immediately",
                "priority": 0,
                "risk": "high",
                "cost": "$5000"
            })

        # Priority-based decision
        if priority == "cost" and delay_min >= 120 and routes.get("alternates"):
            # Cost priority - look for cost-effective reroute
            best_route = min(routes["alternates"], key=lambda x: x.get("extra_cost", 0))
            recommendation.append({
                "action": f"Reroute via {best_route['route']} (+{best_route['extra_time_min']} min)",
                "priority": 1,
                "risk": "medium",
                "cost": f"${best_route['extra_cost']}"
            })
            recommendation.append({
                "action": f"Notify passengers of route change",
                "priority": 2,
                "risk": "low",
                "cost": "$1000"
            })
            total_cost = best_route["extra_cost"] + 1000 + passenger.get("total_cost", 0)

        elif priority == "passenger" and delay_min > 0:
            # Passenger priority - focus on amenities and communication
            amenities_cost = "$3150" if delay_min > 90 else "$900"
            recommendation.append({
                "action": f"Delay {delay_min} min with passenger amenities",
                "priority": 1,
                "risk": risk_level,
                "cost": amenities_cost
            })
            recommendation.append({
                "action": f"Notify passengers: Delay {delay_min} min",
                "priority": 2,
                "risk": "low",
                "cost": "$100"
            })
            recommendation.append({
                "action": "Monitor ATC and provide regular updates",
                "priority": 3,
                "risk": "low",
                "cost": "$0"
            })
            total_cost = (3150 if delay_min > 90 else 900) + 100

        elif priority == "safety":
            # Safety priority - conservative approach
            recommendation.append({
                "action": f"Delay {delay_min} min and conduct safety review",
                "priority": 1,
                "risk": risk_level,
                "cost": "$500"
            })
            recommendation.append({
                "action": "Consult with maintenance for any concerns",
                "priority": 2,
                "risk": "low",
                "cost": "$0"
            })
            total_cost = 500 + passenger.get("total_cost", 0)

        else:
            # Balanced approach
            recommendation.append({
                "action": f"Delay {delay_min} min with standard procedures",
                "priority": 1,
                "risk": risk_level,
                "cost": "$0"
            })
            recommendation.append({
                "action": "Monitor situation and update as needed",
                "priority": 2,
                "risk": "low",
                "cost": "$0"
            })
            total_cost = passenger.get("total_cost", 0)

        return json.dumps({
            "recommendation": recommendation,
            "estimated_cost": f"${total_cost}",
            "risk_level": risk_level,
            "type": "legacy",
            "delay_minutes": delay_min
        })

    except Exception as e:
        logger.error(f"Error in legacy decision: {str(e)}")
        return json.dumps({
            "error": f"Legacy decision failed: {str(e)}",
            "type": "legacy_error"
        })

# --- 6. Unified Decision Function ---

def suggest_operations_decision(
    flight_data: str, weather_data: str, crew_data: str, route_data: str,
    passenger_data: str, prediction_data: str, compliance_data: str,
    feedback_data: str, atc_data: str, threshold_data: str, priority: str,
    use_ai: bool = True, model: str = DEFAULT_MODEL
) -> str:
    """
    Unified decision function - uses AI or legacy based on parameter.

    Args:
        use_ai: If True, use GLM-5.2 AI. If False, use legacy rule-based.
        model: The GLM model to use (default: "glm-5.2")
        priority: "cost", "passenger", "safety", or "balanced"
        All other args are JSON strings from various tools.

    Returns:
        JSON string with decision recommendation
    """
    logger.info(f"Decision requested - AI: {use_ai}, Model: {model}, Priority: {priority}")

    if use_ai:
        return suggest_operations_decision_ai(
            flight_data, weather_data, crew_data, route_data,
            passenger_data, prediction_data, compliance_data,
            feedback_data, atc_data, threshold_data, priority,
            model=model
        )
    else:
        return suggest_operations_decision_legacy(
            flight_data, weather_data, crew_data, route_data,
            passenger_data, prediction_data, compliance_data,
            feedback_data, atc_data, threshold_data, priority
        )

# --- 7. Available Functions Dictionary (for backward compatibility) ---

available_functions = {
    "get_flight_status": get_flight_status,
    "get_aviation_weather": get_aviation_weather,
    "check_crew_availability": check_crew_availability,
    "estimate_passenger_impact": estimate_passenger_impact,
    "predict_delay_escalation": predict_delay_escalation,
    "check_regulatory_compliance": check_regulatory_compliance,
    "estimate_passenger_feedback": estimate_passenger_feedback,
    "get_atc_update": get_atc_update,
    "calculate_cost_threshold": calculate_cost_threshold,
    "pre_plan_crew_swap": pre_plan_crew_swap,
    "get_alternate_routes": get_alternate_routes,
    "update_passenger_feedback": update_passenger_feedback
}

# --- 8. Comparison Function ---

def compare_decisions(
    flight_data: str, weather_data: str, crew_data: str, route_data: str,
    passenger_data: str, prediction_data: str, compliance_data: str,
    feedback_data: str, atc_data: str, threshold_data: str, priority: str
) -> Dict[str, Any]:
    """
    Compare AI and legacy decisions.

    Returns:
        Dictionary with comparison results
    """
    logger.info("Comparing AI and legacy decisions")

    result = {
        "priority": priority,
        "timestamp": datetime.now().isoformat(),
        "ai_decision": None,
        "legacy_decision": None,
        "comparison": {}
    }

    try:
        # Get AI decision
        ai_result = suggest_operations_decision(
            flight_data, weather_data, crew_data, route_data,
            passenger_data, prediction_data, compliance_data,
            feedback_data, atc_data, threshold_data,
            priority=priority, use_ai=True
        )
        result["ai_decision"] = safe_json_loads(ai_result, {"error": "Failed to parse AI decision"})

        # Get legacy decision
        legacy_result = suggest_operations_decision(
            flight_data, weather_data, crew_data, route_data,
            passenger_data, prediction_data, compliance_data,
            feedback_data, atc_data, threshold_data,
            priority=priority, use_ai=False
        )
        result["legacy_decision"] = safe_json_loads(legacy_result, {"error": "Failed to parse legacy decision"})

        # Compare key aspects
        ai_data = result["ai_decision"]
        legacy_data = result["legacy_decision"]

        # Extract key metrics for comparison
        ai_primary = ai_data.get("primary_recommendation", "N/A")[:100] if isinstance(ai_data, dict) else "N/A"

        if isinstance(legacy_data, dict) and "recommendation" in legacy_data:
            legacy_recs = legacy_data.get("recommendation", [])
            legacy_primary = legacy_recs[0].get("action", "N/A") if legacy_recs else "N/A"
        else:
            legacy_primary = "N/A"

        # Get risk levels
        ai_risk = "Unknown"
        if isinstance(ai_data, dict) and "risk_assessment" in ai_data:
            risk_assessment = ai_data.get("risk_assessment", {})
            ai_risk = risk_assessment.get("level", "Unknown") if isinstance(risk_assessment, dict) else "Unknown"

        legacy_risk = legacy_data.get("risk_level", "Unknown") if isinstance(legacy_data, dict) else "Unknown"

        # Get costs
        ai_cost = ai_data.get("estimated_cost", "Unknown") if isinstance(ai_data, dict) else "Unknown"
        legacy_cost = legacy_data.get("estimated_cost", "Unknown") if isinstance(legacy_data, dict) else "Unknown"

        # Determine completeness
        ai_completeness = "Comprehensive" if isinstance(ai_data, dict) and all(k in ai_data for k in ['secondary_actions', 'justification', 'timeline']) else "Basic"

        legacy_completeness = "Comprehensive" if isinstance(legacy_data, dict) and len(legacy_data.get("recommendation", [])) > 2 else "Basic"

        result["comparison"] = {
            "ai_primary_action": ai_primary,
            "legacy_primary_action": legacy_primary,
            "ai_risk": ai_risk,
            "legacy_risk": legacy_risk,
            "ai_cost": ai_cost,
            "legacy_cost": legacy_cost,
            "ai_completeness": ai_completeness,
            "legacy_completeness": legacy_completeness
        }

    except Exception as e:
        logger.error(f"Error in comparison: {str(e)}")
        result["error"] = str(e)

    return result

# --- 9. Visualization Function ---

def visualize_comparison(comparison_result: Dict[str, Any]) -> None:
    """
    Create visualization comparing AI and legacy decisions.
    """
    try:
        fig = go.Figure()

        # Prepare data
        categories = ['Risk Level', 'Cost Efficiency', 'Completeness', 'Actionability', 'Overall Quality']

        # Get scores from comparison data or use defaults
        comparison = comparison_result.get('comparison', {})

        # Calculate scores based on actual comparison data
        ai_risk_score = 4 if 'Low' in comparison.get('ai_risk', '') else 3 if 'Medium' in comparison.get('ai_risk', '') else 2
        legacy_risk_score = 4 if 'Low' in comparison.get('legacy_risk', '') else 3 if 'Medium' in comparison.get('legacy_risk', '') else 2

        ai_scores = [
            ai_risk_score,
            4 if 'Comprehensive' in comparison.get('ai_completeness', '') else 3,
            5 if 'Comprehensive' in comparison.get('ai_completeness', '') else 3,
            4,
            4
        ]

        legacy_scores = [
            legacy_risk_score,
            3,
            3 if 'Comprehensive' in comparison.get('legacy_completeness', '') else 2,
            3,
            3
        ]

        # Add traces
        fig.add_trace(go.Scatterpolar(
            r=ai_scores + [ai_scores[0]],
            theta=categories + [categories[0]],
            fill='toself',
            name='AI (GLM-5.2)',
            line_color='green'
        ))

        fig.add_trace(go.Scatterpolar(
            r=legacy_scores + [legacy_scores[0]],
            theta=categories + [categories[0]],
            fill='toself',
            name='Legacy',
            line_color='red'
        ))

        # Update layout
        fig.update_layout(
            polar=dict(
                radialaxis=dict(
                    visible=True,
                    range=[0, 5]
                )),
            showlegend=True,
            title="Decision Quality Comparison: AI vs Legacy",
            width=600,
            height=500
        )

        fig.show()

    except Exception as e:
        logger.error(f"Error in visualization: {str(e)}")
        print(f"Could not create visualization: {e}")

# --- 10. Main Execution ---

def display_result(title: str, data: Dict[str, Any], indent: int = 0) -> None:
    """Display dictionary data in a formatted way."""
    prefix = "  " * indent

    if isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, dict):
                print(f"{prefix}{key.upper()}:")
                display_result("", value, indent + 1)
            elif isinstance(value, list):
                print(f"{prefix}{key.upper()}:")
                for i, item in enumerate(value, 1):
                    if isinstance(item, dict):
                        print(f"{prefix}  {i}.")
                        display_result("", item, indent + 2)
                    else:
                        print(f"{prefix}  {i}. {item}")
            else:
                print(f"{prefix}{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{prefix}{data}")

def main(priority: str = "balanced", use_ai: bool = True, flight_number: str = "AA123"):
    """
    Main execution flow demonstrating the AI-powered aviation agent.

    Args:
        priority: Decision priority ("cost", "passenger", "safety", "balanced")
        use_ai: Use AI decision making
        flight_number: Flight number to analyze
    """
    print("=" * 70)
    print("✈️  AVIATION OPERATIONS AI AGENT")
    print(f"🤖 Powered by {DEFAULT_MODEL}")
    print(f"📋 Priority: {priority.upper()}")
    print(f"🚀 Flight: {flight_number}")
    print("=" * 70)

    try:
        # Step 1: Gather all data
        print(f"\n📊 Analyzing Flight: {flight_number}")
        print("-" * 50)
        print("⏳ Gathering flight data...")

        # Get flight status first
        flight_data = get_flight_status(flight_number)
        flight = safe_json_loads(flight_data)

        if not flight or flight.get('status') == 'unknown':
            print(f"⚠️  Flight {flight_number} not found in database. Using default data.")

        # Get all other data
        departure = flight.get('departure_icao', 'KJFK')
        destination = flight.get('destination_icao', 'KLAX')
        delay_min = flight.get('delay_minutes', 0)

        weather_data = get_aviation_weather(departure)
        crew_data = check_crew_availability(flight_number)
        route_data = get_alternate_routes(flight_number, departure, destination)
        passenger_data = estimate_passenger_impact(flight_number, delay_min)
        prediction_data = predict_delay_escalation(flight_number, delay_min)
        compliance_data = check_regulatory_compliance(flight_data, crew_data)
        feedback_data = estimate_passenger_feedback(flight_number, delay_min)
        atc_data = get_atc_update(flight_number)
        threshold_data = calculate_cost_threshold(flight_data, passenger_data, prediction_data)

        print("✅ Data gathered successfully!")
        print("-" * 50)

        # Step 2: Generate decision
        print(f"\n🤖 Generating {'AI-powered' if use_ai else 'legacy'} recommendation...")
        print(f"   Priority: {priority}")
        print()

        decision_json = suggest_operations_decision(
            flight_data, weather_data, crew_data, route_data,
            passenger_data, prediction_data, compliance_data,
            feedback_data, atc_data, threshold_data,
            priority=priority,
            use_ai=use_ai
        )

        decision = safe_json_loads(decision_json, {})

        # Step 3: Display results
        print("=" * 70)
        print(f"{'🎯 AI RECOMMENDATION' if use_ai else '📋 LEGACY RECOMMENDATION'}")
        print("=" * 70)

        if "error" in decision:
            print(f"\n⚠️ Error: {decision['error']}")
            if "fallback_recommendation" in decision:
                print(f"📋 Fallback: {decision['fallback_recommendation']}")
        else:
            # Display primary recommendation
            print(f"\n📌 PRIMARY ACTION:")
            if isinstance(decision, dict):
                primary = decision.get('primary_recommendation', 'N/A')
                if primary != 'N/A' and len(str(primary)) > 0:
                    print(f"   {primary}")
                else:
                    print("   No primary action specified")

                # Secondary actions
                secondary = decision.get('secondary_actions', [])
                if secondary:
                    print(f"\n🔄 SECONDARY ACTIONS:")
                    for i, action in enumerate(secondary, 1):
                        if isinstance(action, dict):
                            print(f"   {i}. {action.get('action', str(action))}")
                        else:
                            print(f"   {i}. {action}")

                # Risk assessment
                risk = decision.get('risk_assessment', {})
                if isinstance(risk, dict):
                    print(f"\n⚠️ RISK ASSESSMENT:")
                    print(f"   Level: {risk.get('level', 'Unknown')}")
                    if risk.get('rationale') and risk.get('rationale') != 'Not specified':
                        rationale = risk.get('rationale', '')
                        print(f"   Rationale: {rationale[:200]}..." if len(rationale) > 200 else f"   Rationale: {rationale}")

                # Costs
                print(f"\n💰 ESTIMATED COST:")
                estimated_cost = decision.get('estimated_cost', '$0')
                if isinstance(estimated_cost, dict):
                    print(f"   {estimated_cost.get('value', '$0')}")
                else:
                    print(f"   {estimated_cost}")

                # Passenger impact
                passenger_impact = decision.get('passenger_impact', 'N/A')
                if isinstance(passenger_impact, dict):
                    print(f"\n👥 PASSENGER IMPACT:")
                    if passenger_impact.get('current_impact') and passenger_impact.get('current_impact') != 'Not specified':
                        print(f"   Current: {passenger_impact.get('current_impact', 'N/A')}")
                    if passenger_impact.get('expected_impact') and passenger_impact.get('expected_impact') != 'Not specified':
                        print(f"   Expected: {passenger_impact.get('expected_impact', 'N/A')}")
                    if passenger_impact.get('mitigation') and passenger_impact.get('mitigation') != 'Not specified':
                        print(f"   Mitigation: {passenger_impact.get('mitigation', 'N/A')}")
                else:
                    print(f"\n👥 PASSENGER IMPACT: {passenger_impact}")

                # Timeline
                timeline = decision.get('timeline', 'N/A')
                if isinstance(timeline, dict):
                    print(f"\n⏱️ TIMELINE:")
                    for key, value in timeline.items():
                        if value and value != 'Not specified' and key in ['immediate_actions', 'short_term', 'medium_term', 'long_term']:
                            print(f"   {key.replace('_', ' ').title()}: {value}")
                else:
                    print(f"\n⏱️ TIMELINE: {timeline}")

                # Justification
                if 'justification' in decision and decision['justification']:
                    justification = decision['justification']
                    if isinstance(justification, str) and justification != 'Not specified':
                        print(f"\n📝 JUSTIFICATION:")
                        print(f"   {justification[:300]}..." if len(justification) > 300 else f"   {justification}")

                # Alternatives considered
                alternatives = decision.get('alternatives_considered', [])
                if alternatives:
                    print(f"\n🔄 ALTERNATIVES CONSIDERED:")
                    for alt in alternatives:
                        if isinstance(alt, str):
                            print(f"   • {alt}")
                        elif isinstance(alt, dict):
                            print(f"   • {alt.get('action', alt.get('route', str(alt)))}")

                # Raw response if available and no parsed data
                if 'raw_response' in decision and not any([decision.get('primary_recommendation'),
                                                           decision.get('secondary_actions'),
                                                           decision.get('justification') != 'Not specified']):
                    print(f"\n📄 RAW AI RESPONSE:")
                    print(f"   {decision['raw_response'][:500]}...")

                # Metadata
                print(f"\n📊 METADATA:")
                print(f"   Model: {decision.get('model_used', 'GLM-5.2')}")
                print(f"   Priority: {decision.get('priority', priority)}")
                print(f"   Timestamp: {decision.get('timestamp', datetime.now().isoformat())}")
                if 'delay_minutes' in decision:
                    print(f"   Delay: {decision['delay_minutes']} minutes")
                if decision.get('fallback', False):
                    print(f"   ⚠️ Note: This is a fallback recommendation")

            else:
                print(f"   {decision}")

        print("\n" + "=" * 70)

        # Step 4: Compare with other approach
        print(f"\n🔄 Generating comparison with {'legacy' if use_ai else 'AI'} approach...")
        comparison_result = compare_decisions(
            flight_data, weather_data, crew_data, route_data,
            passenger_data, prediction_data, compliance_data,
            feedback_data, atc_data, threshold_data,
            priority=priority
        )

        comparison = comparison_result.get('comparison', {})
        if comparison:
            print("\n📊 COMPARISON SUMMARY:")
            print("-" * 50)

            # Compare primary actions
            ai_action = comparison.get('ai_primary_action', 'N/A')
            legacy_action = comparison.get('legacy_primary_action', 'N/A')

            print(f"\n   PRIMARY ACTIONS:")
            print(f"   • AI:      {ai_action[:100]}..." if len(ai_action) > 100 else f"   • AI:      {ai_action}")
            print(f"   • Legacy:  {legacy_action[:100]}..." if len(legacy_action) > 100 else f"   • Legacy:  {legacy_action}")

            # Compare risk and cost
            print(f"\n   RISK LEVEL:")
            print(f"   • AI:      {comparison.get('ai_risk', 'Unknown')}")
            print(f"   • Legacy:  {comparison.get('legacy_risk', 'Unknown')}")

            print(f"\n   ESTIMATED COST:")
            print(f"   • AI:      {comparison.get('ai_cost', 'Unknown')}")
            print(f"   • Legacy:  {comparison.get('legacy_cost', 'Unknown')}")

            print(f"\n   COMPLETENESS:")
            print(f"   • AI:      {comparison.get('ai_completeness', 'Unknown')}")
            print(f"   • Legacy:  {comparison.get('legacy_completeness', 'Unknown')}")

            # Comparison verdict
            if use_ai:
                print("\n   ✅ AI RECOMMENDATION PROVIDES:")
                print("   • More comprehensive analysis")
                print("   • Detailed passenger impact assessment")
                print("   • Specific timeline with milestones")
                print("   • Consideration of alternatives")
                print("   • Clear justification for decisions")
            else:
                print("\n   ✅ LEGACY RECOMMENDATION PROVIDES:")
                print("   • Fast, deterministic decisions")
                print("   • Simpler rule-based logic")
                print("   • Predictable outcomes")
                print("   • Lower computational cost")

        # Step 5: Visualization
        if comparison:
            try:
                print("\n📊 Generating comparison visualization...")
                visualize_comparison(comparison_result)
            except Exception as e:
                print(f"   Visualization not available: {e}")

        print("\n" + "=" * 70)
        print("✅ Analysis complete!")
        print("=" * 70)

        # Step 6: Recommendations for next steps
        print("\n💡 RECOMMENDED NEXT STEPS:")
        print("   1. Review the AI recommendation in detail")
        print("   2. Verify key assumptions with actual data")
        print("   3. Consult with operations team for final approval")
        print("   4. Implement recommendation with clear communication")
        print("   5. Monitor outcomes and adjust as needed")
        print("   6. Provide feedback to improve future recommendations")
        print("\n" + "=" * 70)

    except Exception as e:
        logger.error(f"Main execution failed: {str(e)}", exc_info=True)
        print(f"\n❌ Error: {str(e)}")
        print("Please check the logs for more details.")
        print(f"Log file: aoc_agent.log")

# --- 11. Get Tool Definitions Function ---

def get_tool_definitions() -> List[Dict]:
    """Get all tool definitions for OpenAI function calling."""
    return tool_registry.get_tool_definitions()

def get_available_functions() -> Dict[str, callable]:
    """Get all available functions."""
    return available_functions

# --- 12. Additional Utility Functions ---

def analyze_multiple_flights(flights: List[str], priority: str = "balanced"):
    """Analyze multiple flights and generate decisions."""
    results = {}

    for flight in flights:
        print(f"\n{'='*70}")
        print(f"Analyzing Flight: {flight}")
        print(f"{'='*70}")

        try:
            main(priority=priority, flight_number=flight)
            results[flight] = "Success"
        except Exception as e:
            print(f"❌ Failed to analyze {flight}: {e}")
            results[flight] = {"error": str(e)}

    return results

def get_tool_help():
    """Get help information for all registered tools."""
    print("\n📚 AVAILABLE TOOLS:")
    print("-" * 50)

    for name in tool_registry.list_tools():
        metadata = tool_registry.get_metadata(name)
        print(f"\n🔧 {name}")
        print(f"   Category: {metadata.get('category', 'general')}")
        print(f"   Description: {metadata.get('description', 'No description')}")
        print(f"   Registered: {metadata.get('registered_at', 'Unknown')}")

def run_demo():
    """Run a comprehensive demo of the system."""
    print("\n🚀 AVIATION OPS AGENT DEMO")
    print("=" * 70)

    # Test all tools
    print("\n1. TESTING TOOLS")
    print("-" * 50)

    test_flight = "AA123"
    tools_to_test = [
        ("get_flight_status", test_flight),
        ("get_aviation_weather", "KJFK"),
        ("check_crew_availability", test_flight),
        ("estimate_passenger_impact", test_flight, 45),
        ("predict_delay_escalation", test_flight, 45),
    ]

    for tool_info in tools_to_test:
        tool_name = tool_info[0]
        try:
            tool_func = tool_registry.get(tool_name)
            if tool_func:
                if len(tool_info) == 2:
                    result = tool_func(tool_info[1])
                else:
                    result = tool_func(tool_info[1], tool_info[2])
                data = safe_json_loads(result, {})
                print(f"✅ {tool_name}: {len(result)} bytes, status: {data.get('status', 'ok') if isinstance(data, dict) else 'ok'}")
            else:
                print(f"❌ {tool_name}: Tool not found")
        except Exception as e:
            print(f"❌ {tool_name}: {str(e)[:50]}")

    print("\n2. RUNNING FULL ANALYSIS")
    print("-" * 50)

    # Run main with different priorities
    for priority in ["cost", "passenger", "safety", "balanced"]:
        print(f"\nPriority: {priority.upper()}")
        print("-" * 40)
        main(priority=priority, flight_number=test_flight, use_ai=True)
        time.sleep(1)  # Brief pause between runs

    print("\n3. AVAILABLE TOOLS")
    get_tool_help()

    print("\n4. TOOL DEFINITIONS (OpenAI Function Calling Format)")
    print("-" * 50)
    definitions = get_tool_definitions()
    print(json.dumps(definitions, indent=2)[:1000] + "...")

    print("\n" + "=" * 70)
    print("✅ DEMO COMPLETE")
    print("=" * 70)

# --- 13. Main Entry Point ---

if __name__ == "__main__":
    # Parse command line arguments
    import sys

    if len(sys.argv) > 1:
        if sys.argv[1] == "--demo":
            run_demo()
        elif sys.argv[1] == "--help":
            print("Usage: python aoc_agent.py [OPTIONS]")
            print("Options:")
            print("  --demo          Run comprehensive demo")
            print("  --help          Show this help message")
            print("  --flight FCODE  Analyze specific flight (e.g., --flight AA123)")
            print("  --priority P    Set priority (cost, passenger, safety, balanced)")
            print("  --legacy        Use legacy decision instead of AI")
            print("  --tools         List all available tools")
            print("  --definitions   Show tool definitions (OpenAI function calling format)")
            sys.exit(0)
        elif sys.argv[1] == "--tools":
            get_tool_help()
            sys.exit(0)
        elif sys.argv[1] == "--definitions":
            definitions = get_tool_definitions()
            print(json.dumps(definitions, indent=2))
            sys.exit(0)
        else:
            # Parse custom arguments
            flight = "AA123"
            priority = "balanced"
            use_ai = True

            for i, arg in enumerate(sys.argv):
                if arg == "--flight" and i + 1 < len(sys.argv):
                    flight = sys.argv[i + 1]
                elif arg == "--priority" and i + 1 < len(sys.argv):
                    priority = sys.argv[i + 1]
                elif arg == "--legacy":
                    use_ai = False

            main(priority=priority, use_ai=use_ai, flight_number=flight)
    else:
        # Default execution
        main(priority="balanced", use_ai=True, flight_number="AA123")

✈️  AVIATION OPERATIONS AI AGENT
🤖 Powered by glm-5.2
📋 Priority: BALANCED
🚀 Flight: AA123

📊 Analyzing Flight: AA123
--------------------------------------------------
⏳ Gathering flight data...
✅ Data gathered successfully!
--------------------------------------------------

🤖 Generating AI-powered recommendation...
   Priority: balanced

🎯 AI RECOMMENDATION

📌 PRIMARY ACTION:
   Not specified

⚠️ RISK ASSESSMENT:
   Level: Medium

💰 ESTIMATED COST:
   Not specified

👥 PASSENGER IMPACT:

⏱️ TIMELINE:

📊 METADATA:
   Model: glm-5.2
   Priority: balanced
   Timestamp: 2026-06-24T16:10:56.380425
   Delay: 45 minutes


🔄 Generating comparison with legacy approach...

📊 COMPARISON SUMMARY:
--------------------------------------------------

   PRIMARY ACTIONS:
   • AI:      Not specified
   • Legacy:  Delay 45 min with standard procedures

   RISK LEVEL:
   • AI:      Medium
   • Legacy:  medium

   ESTIMATED COST:
   • AI:      Not specified
   • Legacy:  $0

   COMPLETENESS:
   • AI:   


✅ Analysis complete!

💡 RECOMMENDED NEXT STEPS:
   1. Review the AI recommendation in detail
   2. Verify key assumptions with actual data
   3. Consult with operations team for final approval
   4. Implement recommendation with clear communication
   5. Monitor outcomes and adjust as needed
   6. Provide feedback to improve future recommendations



## PART2

In [3]:
import json
import time
import logging
import random
import re
import os
from typing import Dict, List, Any, Optional, Tuple
from datetime import datetime
from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import plotly.graph_objects as go
from collections import defaultdict

# --- 0. Configuration (Logging & API Keys) ---
logging.basicConfig(
    filename='aoc_agent.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Suppress some warnings
import warnings
warnings.filterwarnings('ignore')

# --- Environment Validation ---
def validate_environment():
    """Validate all required environment variables and dependencies."""
    try:
        from google.colab import userdata
        ZAI_KEY_API = userdata.get('ZAI_KEY')
        if not ZAI_KEY_API or ZAI_KEY_API == "PLACEHOLDER_ERROR_KEY":
            raise ValueError("ZAI_KEY not properly configured in Colab Secrets.")
        logger.info("Environment validation: ZAI_KEY found")
        return ZAI_KEY_API
    except ImportError:
        # Not in Colab, try environment variable
        ZAI_KEY_API = os.getenv('ZAI_KEY')
        if not ZAI_KEY_API:
            raise EnvironmentError("ZAI_KEY environment variable not set.")
        logger.info("Environment validation: ZAI_KEY found in environment")
        return ZAI_KEY_API
    except Exception as e:
        logger.error(f"Environment validation failed: {str(e)}")
        raise

# Initialize API Key
try:
    ZAI_KEY_API = validate_environment()
except Exception as e:
    logger.error(f"Fatal: {str(e)}")
    print(f"❌ Fatal Error: {str(e)}")
    print("Please ensure ZAI_KEY is set in Colab Secrets or environment variables.")
    ZAI_KEY_API = None

if not ZAI_KEY_API:
    raise SystemExit("Cannot proceed without valid API key.")

# Initialize Z.ai Client with GLM-5.2 model
client = OpenAI(
    api_key=ZAI_KEY_API,
    base_url="https://api.z.ai/api/paas/v4/"
)

# --- Constants ---
DEFAULT_MODEL = "glm-5.2"
TOOL_TIMEOUT = 30  # seconds
MAX_RETRIES = 3

# --- 1. Tool Registry ---

class ToolRegistry:
    """Registry for all available tools."""

    def __init__(self):
        self._tools = {}
        self._metadata = {}
        self._tool_definitions = []

    def register(self, name: str, func: callable, description: str = "", category: str = "general",
                 parameters: Dict = None, required_params: List[str] = None):
        """Register a tool function with its definition."""
        self._tools[name] = func
        self._metadata[name] = {
            "description": description,
            "category": category,
            "registered_at": datetime.now().isoformat(),
            "parameters": parameters or {},
            "required_params": required_params or []
        }

        # Build tool definition for OpenAI function calling
        if parameters:
            tool_def = {
                "type": "function",
                "function": {
                    "name": name,
                    "description": description,
                    "parameters": {
                        "type": "object",
                        "properties": parameters,
                        "required": required_params or []
                    }
                }
            }
            self._tool_definitions.append(tool_def)

        logger.info(f"Registered tool: {name} ({category})")

    def get(self, name: str) -> Optional[callable]:
        """Get a tool by name."""
        return self._tools.get(name)

    def list_tools(self) -> List[str]:
        """List all registered tool names."""
        return list(self._tools.keys())

    def get_metadata(self, name: str) -> Dict:
        """Get metadata for a tool."""
        return self._metadata.get(name, {})

    def get_tools_by_category(self, category: str) -> List[str]:
        """Get tools by category."""
        return [name for name, meta in self._metadata.items()
                if meta.get("category") == category]

    def get_tool_definitions(self) -> List[Dict]:
        """Get OpenAI function calling definitions for all tools."""
        return self._tool_definitions

    def get_tool_definition(self, name: str) -> Optional[Dict]:
        """Get OpenAI function calling definition for a specific tool."""
        for tool_def in self._tool_definitions:
            if tool_def["function"]["name"] == name:
                return tool_def
        return None

# Initialize global tool registry
tool_registry = ToolRegistry()

# --- 2. Core Tool Definitions ---

def validate_json_string(data: str) -> bool:
    """Validate if string is proper JSON."""
    try:
        json.loads(data)
        return True
    except (json.JSONDecodeError, TypeError):
        return False

def safe_json_loads(data: str, default: Any = None) -> Any:
    """Safely load JSON with fallback."""
    if not data:
        return default or {}
    try:
        return json.loads(data)
    except json.JSONDecodeError as e:
        logger.warning(f"JSON parsing failed: {e}")
        return default or {}

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10),
       retry=retry_if_exception_type(Exception))
def get_flight_status(flight_number: str, departure_icao: str = None) -> str:
    """Retrieve real-time flight status by flight number."""
    logger.info(f"Fetching flight status for {flight_number}")

    if not flight_number or not isinstance(flight_number, str):
        raise ValueError(f"Invalid flight_number: {flight_number}")

    mock_data = {
        "AA123": {
            "flight_number": "AA123",
            "status": "delayed",
            "delay_minutes": 45,
            "eta": "2025-10-01T18:30:00Z",
            "gate": "B12",
            "aircraft": "B737",
            "departure_icao": "KJFK",
            "destination_icao": "KLAX"
        },
        "UA456": {
            "flight_number": "UA456",
            "status": "on_time",
            "delay_minutes": 0,
            "eta": "2025-10-01T17:45:00Z",
            "gate": "C5",
            "aircraft": "A320",
            "departure_icao": "KORD",
            "destination_icao": "KSFO"
        },
        "DL789": {
            "flight_number": "DL789",
            "status": "delayed",
            "delay_minutes": 60,
            "eta": "2025-10-01T19:00:00Z",
            "gate": "A10",
            "aircraft": "A321",
            "departure_icao": "KJFK",
            "destination_icao": "KORD"
        }
    }

    data = mock_data.get(flight_number, {
        "flight_number": flight_number,
        "status": "unknown",
        "delay_minutes": 0,
        "eta": "Unknown",
        "gate": "Unknown",
        "aircraft": "Unknown",
        "departure_icao": departure_icao or "Unknown",
        "destination_icao": "Unknown"
    })

    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def get_aviation_weather(icao_code: str) -> str:
    """Fetch weather data for an airport by ICAO code."""
    logger.info(f"Fetching weather for {icao_code}")

    if not isinstance(icao_code, str) or not icao_code:
        logger.warning(f"Invalid icao_code: {icao_code}, defaulting to KJFK")
        icao_code = "KJFK"

    mock_data = {
        "KJFK": {
            "icao": "KJFK",
            "metar": "KJFK 011730Z 27010KT 10SM FEW030 BKN050 15/10 A2992",
            "conditions": "Light wind, clear skies",
            "temperature": 15,
            "wind_speed": 10,
            "visibility": 10
        },
        "KLAX": {
            "icao": "KLAX",
            "metar": "KLAX 011745Z 18008KT 5SM HZ BKN040 22/18 A2985",
            "conditions": "Haze, moderate visibility",
            "temperature": 22,
            "wind_speed": 8,
            "visibility": 5
        },
        "KORD": {
            "icao": "KORD",
            "metar": "KORD 011800Z 30012KT 8SM OVC035 12/08 A2990",
            "conditions": "Overcast, windy",
            "temperature": 12,
            "wind_speed": 12,
            "visibility": 8
        }
    }

    data = mock_data.get(icao_code, {
        "icao": icao_code,
        "metar": f"{icao_code} 011730Z 27010KT 10SM CLR 20/15 A2995",
        "conditions": "Assumed clear conditions (fallback)",
        "temperature": 20,
        "wind_speed": 10,
        "visibility": 10
    })
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def check_crew_availability(flight_number: str) -> str:
    """Check crew availability for a flight."""
    logger.info(f"Checking crew availability for {flight_number}")

    mock_data = {
        "AA123": {
            "available": True,
            "pilot": "Available (ID: P001, Rest: 10h)",
            "copilot": "Available (ID: C001, Rest: 9h)",
            "cabin_crew": ["Available (ID: CC001, Rest: 8h)", "Available (ID: CC002, Rest: 8h)"],
            "total_crew": 4,
            "rest_compliance": True
        },
        "UA456": {
            "available": False,
            "pilot": "Unavailable (ID: P002, Rest: 2h)",
            "copilot": "Available (ID: C002, Rest: 7h)",
            "cabin_crew": ["Available (ID: CC003, Rest: 6h)"],
            "total_crew": 2,
            "rest_compliance": False
        }
    }

    data = mock_data.get(flight_number, {
        "available": False,
        "pilot": "Unknown",
        "copilot": "Unknown",
        "cabin_crew": [],
        "total_crew": 0,
        "rest_compliance": False
    })
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def estimate_passenger_impact(flight_number: str, delay_minutes: int) -> str:
    """Estimate passenger impact due to delay."""
    logger.info(f"Estimating passenger impact for {flight_number}, delay: {delay_minutes}")

    passengers = random.randint(100, 200)
    rebooking_cost = 200 * passengers if delay_minutes > 60 else 0
    compensation_cost = 50 * passengers if delay_minutes > 120 else 0
    meal_vouchers = 15 * passengers if delay_minutes > 90 else 0
    hotel_cost = 150 * passengers if delay_minutes > 180 else 0

    total_cost = rebooking_cost + compensation_cost + meal_vouchers + hotel_cost

    data = {
        "flight_number": flight_number,
        "passengers": passengers,
        "rebooking_cost": rebooking_cost,
        "compensation_cost": compensation_cost,
        "meal_vouchers": meal_vouchers,
        "hotel_cost": hotel_cost,
        "total_cost": total_cost
    }
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def predict_delay_escalation(flight_number: str, delay_minutes: int) -> str:
    """Predict likelihood of delay escalation."""
    logger.info(f"Predicting delay escalation for {flight_number}, delay: {delay_minutes}")

    base_likelihood = min(0.1 + delay_minutes * 0.005, 0.9)
    hour = datetime.now().hour
    time_factor = 1.0
    if 6 <= hour < 9 or 16 <= hour < 19:
        time_factor = 1.3
    elif 22 <= hour or hour < 5:
        time_factor = 0.7

    likelihood = min(base_likelihood * time_factor, 0.95)
    additional_delay = int(max(15, delay_minutes * 0.5 * (1 + likelihood)))

    data = {
        "escalation_likelihood": round(likelihood, 3),
        "predicted_additional_delay": additional_delay,
        "time_factor": time_factor,
        "confidence": 0.85 - (delay_minutes / 1000)
    }
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def check_regulatory_compliance(flight_data: str, crew_data: str) -> str:
    """Check compliance with aviation regulations."""
    logger.info("Checking regulatory compliance")

    try:
        flight = safe_json_loads(flight_data)
        crew = safe_json_loads(crew_data)

        issues = []
        warnings = []

        if not crew.get("available", False):
            issues.append("Crew rest non-compliant")
        if not crew.get("rest_compliance", False):
            issues.append("Crew rest period insufficient")

        delay = flight.get("delay_minutes", 0)
        if delay > 180:
            issues.append("Delay exceeds regulatory threshold (180 min)")
        elif delay > 120:
            warnings.append("Delay approaching regulatory threshold (180 min)")

        if crew.get("total_crew", 0) < 3:
            warnings.append("Minimum crew complement may be insufficient")

        data = {
            "status": "Non-compliant" if issues else "Compliant with warnings" if warnings else "Fully Compliant",
            "issues": issues,
            "warnings": warnings,
            "compliance_score": 1.0 - (len(issues) * 0.3) - (len(warnings) * 0.1)
        }
        return json.dumps(data)

    except Exception as e:
        logger.error(f"Error in check_regulatory_compliance: {str(e)}")
        return json.dumps({"error": str(e), "status": "Unknown"})

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def estimate_passenger_feedback(flight_number: str, delay_minutes: int) -> str:
    """Estimate passenger feedback based on delay."""
    logger.info(f"Estimating passenger feedback for {flight_number}, delay: {delay_minutes}")

    satisfaction = max(0.1, 0.9 - delay_minutes * 0.005)
    satisfaction = min(0.99, satisfaction + random.uniform(-0.05, 0.05))

    if satisfaction > 0.75:
        feedback = "Positive"
        sentiment_score = 0.8
    elif satisfaction > 0.5:
        feedback = "Neutral"
        sentiment_score = 0.5
    elif satisfaction > 0.3:
        feedback = "Negative"
        sentiment_score = 0.2
    else:
        feedback = "Very Negative"
        sentiment_score = 0.05

    data = {
        "flight_number": flight_number,
        "satisfaction_score": round(satisfaction, 3),
        "feedback": feedback,
        "sentiment_score": sentiment_score,
        "delay_impact": "High" if delay_minutes > 120 else "Medium" if delay_minutes > 60 else "Low"
    }
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def get_atc_update(flight_number: str) -> str:
    """Fetch ATC updates for a flight."""
    logger.info(f"Fetching ATC update for {flight_number}")

    atc_responses = [
        {"status": "Clearance expected in 30 min", "additional_delay_min": 30, "confidence": "High"},
        {"status": "Holding pattern expected", "additional_delay_min": 45, "confidence": "Medium"},
        {"status": "Immediate clearance available", "additional_delay_min": 0, "confidence": "High"},
        {"status": "Weather delay at destination", "additional_delay_min": 60, "confidence": "Low"}
    ]

    data = random.choice(atc_responses)
    data["flight_number"] = flight_number
    data["timestamp"] = datetime.now().isoformat()

    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def calculate_cost_threshold(flight_data: str, passenger_data: str, prediction_data: str) -> str:
    """Calculate cost thresholds for delay vs. cancellation."""
    logger.info("Calculating cost threshold")

    try:
        flight = safe_json_loads(flight_data)
        passenger = safe_json_loads(passenger_data)
        prediction = safe_json_loads(prediction_data)

        delay_min = flight.get("delay_minutes", 0)
        passengers = passenger.get("passengers", 100)

        fuel_cost_per_min = 150
        crew_cost_per_hour = 500

        delay_cost = (delay_min * fuel_cost_per_min) + \
                    (delay_min / 60 * crew_cost_per_hour * 3) + \
                    passenger.get("total_cost", 0)

        cancel_cost = 50000 + (passengers * 350)

        escalation_extra = prediction.get("predicted_additional_delay", 0) * 150
        delay_cost_with_escalation = delay_cost + escalation_extra

        avg_ticket_price = 450
        revenue_loss = passengers * avg_ticket_price * 0.3

        total_cancel_cost = cancel_cost + revenue_loss

        data = {
            "delay_cost": int(delay_cost),
            "delay_cost_with_escalation": int(delay_cost_with_escalation),
            "cancel_cost": int(total_cancel_cost),
            "cost_difference": int(total_cancel_cost - delay_cost),
            "threshold_min": 120,
            "current_delay": delay_min,
            "recommendation": "Cancel" if delay_min > 120 or (delay_min > 60 and prediction.get("escalation_likelihood", 0) > 0.7) else "Delay",
            "confidence": min(0.95, 0.8 + (1 - prediction.get("escalation_likelihood", 0)))
        }
        return json.dumps(data)

    except Exception as e:
        logger.error(f"Error in calculate_cost_threshold: {str(e)}")
        return json.dumps({"error": str(e)})

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def pre_plan_crew_swap(flight_number: str, crew_data: str, delay_minutes: int) -> str:
    """Pre-plan crew swap if needed."""
    logger.info(f"Pre-planning crew swap for {flight_number}, delay: {delay_minutes}")

    try:
        crew = safe_json_loads(crew_data)

        need_swap = not crew.get("available", False)

        if delay_minutes > 120 and crew.get("available", False):
            pilot_rest = crew.get("pilot", "")
            if "Rest: " in pilot_rest:
                rest_hours = int(re.search(r'Rest: (\d+)h', pilot_rest).group(1)) if re.search(r'Rest: (\d+)h', pilot_rest) else 10
                if rest_hours < 4:
                    need_swap = True

        recommendation = "Swap required" if need_swap else "No swap needed"

        data = {
            "recommendation": recommendation,
            "need_swap": need_swap,
            "standby_crew": None if not need_swap else {
                "pilot": "P002 (Available, 12h rest)",
                "copilot": "C002 (Available, 11h rest)",
                "cabin_crew": ["CC003 (Available, 10h rest)", "CC004 (Available, 9h rest)"]
            },
            "swap_duration_min": 30 if need_swap else 0,
            "swap_cost": 2000 if need_swap else 0
        }
        return json.dumps(data)

    except Exception as e:
        logger.error(f"Error in pre_plan_crew_swap: {str(e)}")
        return json.dumps({"error": str(e), "recommendation": "Unable to assess"})

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def get_alternate_routes(flight_number: str, departure_icao: str, destination_icao: str) -> str:
    """Fetch alternate routes for a flight."""
    logger.info(f"Fetching alternate routes for {flight_number} from {departure_icao} to {destination_icao}")

    alternates = {
        "KJFK-KLAX": [
            {"route": "KJFK-KPHL-KLAX", "extra_time_min": 20, "extra_cost": 3000, "extra_fuel": 1500, "risk": "low"},
            {"route": "KJFK-KORD-KLAX", "extra_time_min": 45, "extra_cost": 5000, "extra_fuel": 3000, "risk": "medium"},
            {"route": "KJFK-KDFW-KLAX", "extra_time_min": 60, "extra_cost": 6500, "extra_fuel": 4000, "risk": "medium"}
        ],
        "KORD-KSFO": [
            {"route": "KORD-KDEN-KSFO", "extra_time_min": 25, "extra_cost": 3500, "extra_fuel": 2000, "risk": "low"},
            {"route": "KORD-KSLC-KSFO", "extra_time_min": 40, "extra_cost": 4500, "extra_fuel": 2800, "risk": "medium"}
        ]
    }

    route_key = f"{departure_icao}-{destination_icao}"
    alt_routes = alternates.get(route_key, [
        {"route": f"{departure_icao}-KATL-{destination_icao}", "extra_time_min": 30, "extra_cost": 4000, "extra_fuel": 2500, "risk": "low"},
        {"route": f"{departure_icao}-KDEN-{destination_icao}", "extra_time_min": 50, "extra_cost": 5500, "extra_fuel": 3500, "risk": "medium"}
    ])

    data = {
        "original_route": f"{departure_icao}-{destination_icao} direct",
        "alternates": alt_routes,
        "weather_impacts": {
            "KPHL": "Clear",
            "KORD": "Light rain",
            "KDEN": "Snow possible"
        }
    }
    return json.dumps(data)

@retry(stop=stop_after_attempt(MAX_RETRIES),
       wait=wait_exponential(multiplier=1, min=2, max=10))
def update_passenger_feedback(flight_number: str, satisfaction_score: float, action: str) -> str:
    """Update passenger feedback based on action taken."""
    logger.info(f"Updating passenger feedback for {flight_number}, action: {action}")

    positive_actions = ["amenities", "voucher", "upgrade", "compensation", "communication"]
    negative_actions = ["delay", "cancel", "deny"]

    impact = 0
    action_lower = action.lower()

    for pos in positive_actions:
        if pos in action_lower:
            impact += 0.1
    for neg in negative_actions:
        if neg in action_lower:
            impact -= 0.2

    new_score = min(0.95, max(0.05, satisfaction_score + impact))
    new_score = min(0.95, max(0.05, new_score + random.uniform(-0.02, 0.02)))

    if new_score > 0.75:
        feedback = "Positive"
    elif new_score > 0.5:
        feedback = "Neutral"
    elif new_score > 0.25:
        feedback = "Negative"
    else:
        feedback = "Very Negative"

    data = {
        "flight_number": flight_number,
        "previous_satisfaction": round(satisfaction_score, 3),
        "satisfaction_score": round(new_score, 3),
        "feedback": feedback,
        "action_taken": action,
        "improvement": round(new_score - satisfaction_score, 3),
        "timestamp": datetime.now().isoformat()
    }
    return json.dumps(data)

# --- 3. Register All Tools with Definitions ---

# Register each tool with its parameter definitions
tool_registry.register(
    "get_flight_status", get_flight_status,
    description="Retrieve real-time flight status by flight number.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number, e.g., AA123"},
        "departure_icao": {"type": "string", "description": "ICAO code of departure airport, optional"}
    },
    required_params=["flight_number"],
    category="flight_operations"
)

tool_registry.register(
    "get_aviation_weather", get_aviation_weather,
    description="Fetch weather data for an airport by ICAO code.",
    parameters={
        "icao_code": {"type": "string", "description": "ICAO code, e.g., KJFK"}
    },
    required_params=["icao_code"],
    category="weather"
)

tool_registry.register(
    "check_crew_availability", check_crew_availability,
    description="Check crew availability for a flight.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number, e.g., AA123"}
    },
    required_params=["flight_number"],
    category="crew_management"
)

tool_registry.register(
    "estimate_passenger_impact", estimate_passenger_impact,
    description="Estimate passenger impact due to delay.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "delay_minutes": {"type": "integer", "description": "Delay duration in minutes"}
    },
    required_params=["flight_number", "delay_minutes"],
    category="passenger_services"
)

tool_registry.register(
    "predict_delay_escalation", predict_delay_escalation,
    description="Predict likelihood of delay escalation.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "delay_minutes": {"type": "integer", "description": "Current delay in minutes"}
    },
    required_params=["flight_number", "delay_minutes"],
    category="predictions"
)

tool_registry.register(
    "check_regulatory_compliance", check_regulatory_compliance,
    description="Check compliance with aviation regulations.",
    parameters={
        "flight_data": {"type": "string", "description": "JSON string of flight data"},
        "crew_data": {"type": "string", "description": "JSON string of crew data"}
    },
    required_params=["flight_data", "crew_data"],
    category="compliance"
)

tool_registry.register(
    "estimate_passenger_feedback", estimate_passenger_feedback,
    description="Estimate passenger feedback based on delay.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "delay_minutes": {"type": "integer", "description": "Delay duration in minutes"}
    },
    required_params=["flight_number", "delay_minutes"],
    category="passenger_services"
)

tool_registry.register(
    "get_atc_update", get_atc_update,
    description="Fetch ATC updates for a flight.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"}
    },
    required_params=["flight_number"],
    category="flight_operations"
)

tool_registry.register(
    "calculate_cost_threshold", calculate_cost_threshold,
    description="Calculate cost thresholds for delay vs. cancellation.",
    parameters={
        "flight_data": {"type": "string", "description": "JSON string of flight data"},
        "passenger_data": {"type": "string", "description": "JSON string of passenger data"},
        "prediction_data": {"type": "string", "description": "JSON string of prediction data"}
    },
    required_params=["flight_data", "passenger_data", "prediction_data"],
    category="cost_analysis"
)

tool_registry.register(
    "pre_plan_crew_swap", pre_plan_crew_swap,
    description="Pre-plan crew swap if needed.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "crew_data": {"type": "string", "description": "JSON string of crew data"},
        "delay_minutes": {"type": "integer", "description": "Delay duration in minutes"}
    },
    required_params=["flight_number", "crew_data", "delay_minutes"],
    category="crew_management"
)

tool_registry.register(
    "get_alternate_routes", get_alternate_routes,
    description="Fetch alternate routes for a flight.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "departure_icao": {"type": "string", "description": "ICAO code of departure"},
        "destination_icao": {"type": "string", "description": "ICAO code of destination"}
    },
    required_params=["flight_number", "departure_icao", "destination_icao"],
    category="flight_operations"
)

tool_registry.register(
    "update_passenger_feedback", update_passenger_feedback,
    description="Update passenger feedback based on action taken.",
    parameters={
        "flight_number": {"type": "string", "description": "Flight number"},
        "satisfaction_score": {"type": "number", "description": "Current satisfaction score"},
        "action": {"type": "string", "description": "Action taken, e.g., provide amenities"}
    },
    required_params=["flight_number", "satisfaction_score", "action"],
    category="passenger_services"
)

# --- 4. Available Functions Dictionary ---

available_functions = {
    "get_flight_status": get_flight_status,
    "get_aviation_weather": get_aviation_weather,
    "check_crew_availability": check_crew_availability,
    "estimate_passenger_impact": estimate_passenger_impact,
    "predict_delay_escalation": predict_delay_escalation,
    "check_regulatory_compliance": check_regulatory_compliance,
    "estimate_passenger_feedback": estimate_passenger_feedback,
    "get_atc_update": get_atc_update,
    "calculate_cost_threshold": calculate_cost_threshold,
    "pre_plan_crew_swap": pre_plan_crew_swap,
    "get_alternate_routes": get_alternate_routes,
    "update_passenger_feedback": update_passenger_feedback
}

# --- 5. AOCCAgent Class (FIXED) ---

class AOCCAgent:
    def __init__(self):
        self.client = client
        self.conversation_history = []
        self.metrics = {"tools_called": 0, "response_time": 0.0, "errors": []}
        self.context = {
            "flight_number": None,
            "delay_minutes": None,
            "crew_available": None,
            "passenger_cost": 0,
            "escalation_likelihood": 0.5,
            "satisfaction_score": 0.5,
            "atc_status": "No update",
            "atc_delay": 0,
            "threshold_recommendation": "Delay",
            "departure_icao": None,
            "destination_icao": None,
            "priority": "passenger",
            "weather_conditions": "Unknown",
            "compliance_status": "Compliant",
            "alternate_routes": []
        }
        self.tools = tool_registry.get_tool_definitions()
        self.available_functions = available_functions

    def validate_tool_output(self, output, tool_name):
        """Validate tool output is valid JSON."""
        logger.info(f"Validating output for {tool_name}: {str(output)[:100]}...")
        if output is None:
            logger.error(f"Tool {tool_name} returned None")
            self.metrics["errors"].append(f"Tool {tool_name} returned None")
            return json.dumps({"error": f"Tool {tool_name} returned None"})
        if not isinstance(output, str):
            logger.error(f"Tool {tool_name} returned non-string output: {type(output)}")
            self.metrics["errors"].append(f"Tool {tool_name} returned non-string: {type(output)}")
            return json.dumps({"error": f"Tool {tool_name} returned non-string output: {type(output)}"})
        try:
            json.loads(output)
            return output
        except json.JSONDecodeError as e:
            logger.error(f"JSONDecodeError in {tool_name} output: {str(e)}")
            self.metrics["errors"].append(f"JSONDecodeError in {tool_name}: {str(e)}")
            return json.dumps({"error": f"Invalid JSON from {tool_name}: {str(e)}"})

    def update_context(self, tool_outputs: List[Dict[str, Any]], user_prompt: str):
        """Update context from tool outputs and user prompt."""
        for output in tool_outputs:
            data = {}
            if output.get("content") and isinstance(output["content"], str):
                try:
                    data = json.loads(output["content"])
                except json.JSONDecodeError as e:
                    logger.error(f"JSONDecodeError in parsing tool output {output['name']}: {str(e)}")
                    self.metrics["errors"].append(f"JSONDecodeError in {output['name']}: {str(e)}")
                    continue

            # Update context based on tool name
            if output["name"] == "get_flight_status":
                self.context.update({
                    "flight_number": data.get("flight_number"),
                    "delay_minutes": data.get("delay_minutes", 0),
                    "departure_icao": data.get("departure_icao", "KJFK"),
                    "destination_icao": data.get("destination_icao", "KLAX")
                })
            elif output["name"] == "get_aviation_weather":
                self.context.update({
                    "weather_conditions": data.get("conditions", "Unknown"),
                    "temperature": data.get("temperature", "N/A"),
                    "wind_speed": data.get("wind_speed", "N/A"),
                    "visibility": data.get("visibility", "N/A")
                })
            elif output["name"] == "check_crew_availability":
                self.context.update({
                    "crew_available": data.get("available", False),
                    "crew_data": output["content"],
                    "total_crew": data.get("total_crew", 0)
                })
            elif output["name"] == "estimate_passenger_impact":
                self.context.update({
                    "passenger_cost": data.get("total_cost", 0),
                    "passengers": data.get("passengers", 0)
                })
            elif output["name"] == "predict_delay_escalation":
                self.context.update({
                    "escalation_likelihood": data.get("escalation_likelihood", 0.5),
                    "predicted_additional_delay": data.get("predicted_additional_delay", 30)
                })
            elif output["name"] in ["estimate_passenger_feedback", "update_passenger_feedback"]:
                self.context.update({
                    "satisfaction_score": data.get("satisfaction_score", 0.5),
                    "feedback": data.get("feedback", "Neutral")
                })
            elif output["name"] == "get_atc_update":
                self.context.update({
                    "atc_status": data.get("status", "No update"),
                    "atc_delay": data.get("additional_delay_min", 0)
                })
            elif output["name"] == "calculate_cost_threshold":
                self.context.update({
                    "threshold_recommendation": data.get("recommendation", "Delay"),
                    "delay_cost": data.get("delay_cost", 0),
                    "cancel_cost": data.get("cancel_cost", 0)
                })
            elif output["name"] == "check_regulatory_compliance":
                self.context.update({
                    "compliance_status": data.get("status", "Compliant"),
                    "compliance_issues": data.get("issues", [])
                })
            elif output["name"] == "get_alternate_routes":
                self.context.update({
                    "alternate_routes": data.get("alternates", [])
                })
            elif output["name"] == "pre_plan_crew_swap":
                self.context.update({
                    "crew_swap_needed": data.get("need_swap", False)
                })
            elif output["name"] == "suggest_operations_decision":
                # Parse decision data
                if isinstance(data, dict):
                    if "primary_recommendation" in data:
                        self.context["primary_recommendation"] = data.get("primary_recommendation")
                        self.context["secondary_actions"] = data.get("secondary_actions", [])
                        self.context["risk_assessment"] = data.get("risk_assessment", {})
                        self.context["estimated_cost"] = data.get("estimated_cost", "$0")
                        self.context["timeline"] = data.get("timeline", {})
                    elif "recommendation" in data:
                        # Handle legacy format
                        self.context["recommendation"] = data.get("recommendation", [])
                        self.context["estimated_cost"] = data.get("estimated_cost", "$0")
                        self.context["risk_level"] = data.get("risk_level", "medium")

        # Update priority from prompt
        prompt_lower = user_prompt.lower()
        if re.search(r"\bprioritize\s+cost\b|\bminimize\s+cost\b|\bcost\s+effective\b|\bcost\b", prompt_lower):
            self.context["priority"] = "cost"
            logger.info("Updated priority to cost from prompt")
        elif re.search(r"\bprioritize\s+passenger\b|\bpassenger\s+satisfaction\b|\bpassenger\s+experience\b", prompt_lower):
            self.context["priority"] = "passenger"
            logger.info("Updated priority to passenger from prompt")
        elif re.search(r"\bprioritize\s+operation\b|\boperational\b|\befficiency\b", prompt_lower):
            self.context["priority"] = "operation"
            logger.info("Updated priority to operation from prompt")
        elif re.search(r"\bbalanced\b|\bbalance\b", prompt_lower):
            self.context["priority"] = "balanced"
            logger.info("Updated priority to balanced from prompt")
        else:
            logger.info(f"No priority specified, retaining priority: {self.context['priority']}")

        # Update delay_minutes from prompt
        match = re.search(r"(\d+)\s*(?:minute|min)\w*(?:\s*delay)?", prompt_lower)
        if match:
            self.context["delay_minutes"] = int(match.group(1))
            logger.info(f"Updated delay_minutes to {self.context['delay_minutes']} from prompt")
        elif "delay" in prompt_lower and not self.context.get("delay_minutes"):
            self.context["delay_minutes"] = 90
            logger.info("No specific delay found in prompt, defaulting to 90 minutes")

    def safe_float_conversion(self, value: Any, default: float = 0.0) -> float:
        """Safely convert a value to float."""
        if value is None:
            return default
        if isinstance(value, (int, float)):
            return float(value)
        if isinstance(value, str):
            # Remove currency symbols and commas
            cleaned = re.sub(r'[$,]', '', value)
            try:
                return float(cleaned)
            except ValueError:
                return default
        return default

    def reflect(self, tool_outputs: List[Dict[str, Any]], priority: str) -> str:
        """Reflect on tool outputs and synthesize a recommendation."""
        logger.info("Reflecting on tool outputs for synthesis")
        priority = self.context.get("priority", priority)

        # Map priority to valid values
        priority_map = {
            "cost": "cost",
            "passenger": "passenger",
            "operation": "operation",
            "balanced": "balanced"
        }
        priority = priority_map.get(priority, "passenger")
        self.context["priority"] = priority

        recommendation = []
        total_cost = 0
        risk_level = "medium"
        compliance_status = "Compliant"
        passenger_impact = "Neutral"
        satisfaction_score = 0.5
        audit_trail = f"Reflection at {time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}"

        # Extract data from context
        flight_number = self.context.get("flight_number", "AA123")
        delay_min = self.context.get("delay_minutes", 90)
        crew_available = self.context.get("crew_available", False)
        pax_cost = self.context.get("passenger_cost", 0)
        escalation_likelihood = self.context.get("escalation_likelihood", 0.5)
        satisfaction_score = self.context.get("satisfaction_score", 0.5)
        atc_status = self.context.get("atc_status", "No update")
        weather_conditions = self.context.get("weather_conditions", "Unknown")
        alternate_routes = self.context.get("alternate_routes", [])
        compliance_status = self.context.get("compliance_status", "Compliant")

        # Process tool outputs
        for output in tool_outputs:
            try:
                if output.get("content"):
                    content = output["content"]
                    if isinstance(content, str):
                        data = json.loads(content)
                    else:
                        data = content

                    if output["name"] == "get_alternate_routes":
                        alternate_routes = data.get("alternates", [])
                    elif output["name"] == "check_regulatory_compliance":
                        compliance_status = data.get("status", "Compliant")
                    elif output["name"] == "suggest_operations_decision":
                        if "recommendation" in data:
                            recommendation = data.get("recommendation", [])
                            total_cost = self.safe_float_conversion(
                                data.get("estimated_cost", "0").replace("$", "").replace(",", "")
                            )
                            risk_level = data.get("risk_level", "medium")
                        elif "primary_recommendation" in data:
                            # New format
                            recommendation = [{
                                "action": data.get("primary_recommendation", "No action"),
                                "priority": 1,
                                "risk": data.get("risk_assessment", {}).get("level", "medium"),
                                "cost": data.get("estimated_cost", "$0"),
                                "passenger_impact": "Neutral"
                            }]
                            total_cost = self.safe_float_conversion(
                                data.get("estimated_cost", "0").replace("$", "").replace(",", "")
                            )
                            risk_level = data.get("risk_assessment", {}).get("level", "medium").lower()
                    elif output["name"] == "estimate_passenger_impact":
                        pax_cost = data.get("total_cost", 0)
                    elif output["name"] == "predict_delay_escalation":
                        escalation_likelihood = data.get("escalation_likelihood", 0.5)
                    elif output["name"] in ["estimate_passenger_feedback", "update_passenger_feedback"]:
                        satisfaction_score = data.get("satisfaction_score", 0.5)
                    elif output["name"] == "get_aviation_weather":
                        weather_conditions = data.get("conditions", "Unknown")
            except (json.JSONDecodeError, TypeError, ValueError) as e:
                logger.error(f"Error in reflect for {output.get('name', 'unknown')}: {str(e)}")
                self.metrics["errors"].append(f"Error in {output.get('name', 'unknown')}: {str(e)}")
                continue

        # Build action plan based on priority
        actions = []
        if priority == "cost" and delay_min >= 120 and alternate_routes:
            # Cost-optimized plan with rerouting
            best_route = min(alternate_routes, key=lambda x: x.get("extra_cost", float('inf')))
            actions.append({
                "action": f"Reroute via {best_route['route']} (+{best_route['extra_time_min']} min)",
                "priority": 1,
                "risk": "medium",
                "cost": f"${best_route.get('extra_cost', 3000)}",
                "passenger_impact": "Neutral",
                "rationale": f"Minimizes cost while adding only {best_route.get('extra_time_min', 0)} minutes"
            })
            actions.append({
                "action": f"Notify passengers of reroute via {best_route['route']}",
                "priority": 2,
                "risk": "low",
                "cost": "$1500",
                "passenger_impact": "Neutral",
                "rationale": "Proactive communication maintains trust"
            })
            actions.append({
                "action": "Proactive connection management at destination",
                "priority": 3,
                "risk": "medium",
                "cost": f"${pax_cost}",
                "passenger_impact": "Neutral",
                "rationale": "Protects connecting passengers"
            })
            total_cost = best_route.get("extra_cost", 3000) + 1500 + pax_cost
            passenger_impact = "Neutral"

        elif priority == "passenger" and delay_min > 0:
            # Passenger-focused plan with amenities
            amenities_cost = 3150 if delay_min > 90 else 900
            actions.append({
                "action": f"Delay {delay_min} min and provide amenities",
                "priority": 1,
                "risk": "medium" if escalation_likelihood > 0.3 else "low",
                "cost": f"${amenities_cost}",
                "passenger_impact": "Positive",
                "rationale": "Prioritizes passenger comfort and satisfaction"
            })
            actions.append({
                "action": f"Proactive passenger communication: Delay {delay_min} min",
                "priority": 2,
                "risk": "low",
                "cost": "$100",
                "passenger_impact": "Positive",
                "rationale": "Transparent communication improves satisfaction"
            })
            actions.append({
                "action": "Monitor ATC and provide regular updates",
                "priority": 3,
                "risk": "low",
                "cost": "$0",
                "passenger_impact": "Neutral",
                "rationale": "Keeps passengers informed"
            })
            total_cost = amenities_cost + 100
            passenger_impact = "Positive"

        elif priority == "operation":
            # Operational efficiency plan
            actions.append({
                "action": f"Analyze operational impact of {delay_min} min delay",
                "priority": 1,
                "risk": "medium",
                "cost": "$500",
                "passenger_impact": "Neutral",
                "rationale": "Understand cascading effects"
            })
            actions.append({
                "action": "Coordinate with ground operations for efficient turnaround",
                "priority": 2,
                "risk": "low",
                "cost": "$200",
                "passenger_impact": "Neutral",
                "rationale": "Optimizes ground time"
            })
            actions.append({
                "action": "Monitor and adjust crew schedules as needed",
                "priority": 3,
                "risk": "low",
                "cost": "$300",
                "passenger_impact": "Neutral",
                "rationale": "Prevents crew fatigue issues"
            })
            total_cost = 1000

        elif priority == "balanced":
            # Balanced approach
            actions.append({
                "action": f"Delay {delay_min} min with standard procedures",
                "priority": 1,
                "risk": "medium",
                "cost": "$500",
                "passenger_impact": "Neutral",
                "rationale": "Balances all considerations"
            })
            actions.append({
                "action": "Communicate updates to passengers",
                "priority": 2,
                "risk": "low",
                "cost": "$100",
                "passenger_impact": "Positive",
                "rationale": "Maintains transparency"
            })
            actions.append({
                "action": "Monitor situation and re-evaluate if needed",
                "priority": 3,
                "risk": "low",
                "cost": "$0",
                "passenger_impact": "Neutral",
                "rationale": "Allows for adaptive management"
            })
            total_cost = 600
            passenger_impact = "Neutral"
        else:
            # Default/fallback plan
            actions = recommendation if recommendation else [{
                "action": f"Delay {delay_min} min and monitor",
                "priority": 1,
                "risk": "medium",
                "cost": "$0",
                "passenger_impact": "Neutral",
                "rationale": "Conservative approach"
            }]
            total_cost = pax_cost
            passenger_impact = "Neutral"

        # Build final content
        final_content = (
            f"### AOC Action Plan: Flight {flight_number} ({delay_min}-Minute Delay)\n"
            f"**Priority:** {priority.capitalize()}\n\n"
            f"### 1. Executive Summary\n"
            f"Flight {flight_number} delayed by {delay_min} minutes. "
            f"Crew available: {crew_available}. "
            f"Passenger cost: ${pax_cost:,}. "
            f"Escalation likelihood: {escalation_likelihood:.2f}. "
            f"Satisfaction score: {satisfaction_score:.2f}. "
            f"Compliance: {compliance_status}. "
            f"ATC update: {atc_status}. "
            f"Weather: {weather_conditions}.\n"
            f"**Key Decision:** {actions[0]['action'] if actions else 'No action determined'}\n\n"
            f"### 2. Prioritized Action Plan\n"
        )

        for i, action in enumerate(actions, 1):
            final_content += (
                f"#### Action {i}: {action['action']}\n"
                f"* **Priority:** {action['priority']}\n"
                f"* **Risk:** {action['risk']}\n"
                f"* **Cost:** {action['cost']}\n"
                f"* **Passenger Impact:** {action['passenger_impact']}\n"
                f"* **Rationale:** {action.get('rationale', 'N/A')}\n"
            )

        final_content += (
            f"\n### 3. Regulatory Compliance\n"
            f"Status: {compliance_status}. "
            f"No regulatory violations identified.\n\n"
            f"### 4. Assumptions & Context\n"
            f"Delay duration: {delay_min} minutes. "
            f"Priority: {priority}. "
            f"Total estimated cost: ${total_cost:,.0f}. "
            f"Risk level: {risk_level}. "
            f"Passenger impact: {passenger_impact}. "
            f"Audit trail: {audit_trail}\n"
        )

        logger.info("Reflection completed")
        return final_content

    def execute_tool(self, tool_name: str, tool_args: Dict) -> Optional[Dict]:
        """Execute a tool and return the result."""
        func = self.available_functions.get(tool_name)
        if not func:
            logger.error(f"Tool {tool_name} not found in available functions")
            return {"error": f"Tool {tool_name} not found"}

        try:
            # Validate arguments
            tool_def = tool_registry.get_tool_definition(tool_name)
            if tool_def:
                required_params = tool_def["function"]["parameters"]["required"]
                for param in required_params:
                    if param not in tool_args:
                        logger.warning(f"Missing required parameter {param} for {tool_name}")
                        # Try to get from context
                        if param in self.context:
                            tool_args[param] = self.context[param]

            # Execute the tool
            output = func(**tool_args)
            output = self.validate_tool_output(output, tool_name)

            return {
                "tool_call_id": f"manual_{tool_name}",
                "role": "tool",
                "name": tool_name,
                "content": output
            }
        except Exception as e:
            error_msg = json.dumps({"error": f"Tool {tool_name} failed: {str(e)}"})
            self.metrics["errors"].append(f"Tool {tool_name} failed: {str(e)}")
            logger.error(f"Tool {tool_name} failed: {str(e)}")
            return {
                "tool_call_id": f"manual_{tool_name}",
                "role": "tool",
                "name": tool_name,
                "content": error_msg
            }

    def run_query(self, user_prompt: str, max_iterations: int = 2) -> str:
        """Run a query through the AOC agent."""
        if ZAI_KEY_API == "PLACEHOLDER_ERROR_KEY" or not ZAI_KEY_API:
            logger.error("API Key not loaded")
            return "Cannot run agent: API Key not loaded."

        # Reset conversation for new query
        self.conversation_history = []
        self.conversation_history.append({"role": "user", "content": user_prompt})

        start_time = time.time()
        final_content = "No recommendation generated due to processing error."
        priority = self.context.get("priority", "passenger")

        for iteration in range(max_iterations):
            logger.info(f"Starting iteration {iteration + 1}")
            print(f"AOC Agent: Iteration {iteration + 1} - Planning...")

            try:
                response = self.client.chat.completions.create(
                    model="glm-5.2",
                    messages=self.conversation_history,
                    tools=self.tools,
                    tool_choice="auto",
                )
                response_message = response.choices[0].message
                self.conversation_history.append(response_message)
                logger.info("Planning completed")
            except Exception as e:
                self.metrics["errors"].append(f"API call failed: {str(e)}")
                logger.error(f"API call failed: {str(e)}")
                self.conversation_history.append({"role": "assistant", "content": f"API error: {str(e)}. Attempting fallback."})
                continue

            tool_outputs = []

            # Process tool calls from the model
            if response_message.tool_calls:
                print("AOC Agent: Executing tools...")
                self.metrics["tools_called"] += len(response_message.tool_calls)
                for tool_call in response_message.tool_calls:
                    func_name = tool_call.function.name
                    func = self.available_functions.get(func_name)
                    if func:
                        try:
                            args = json.loads(tool_call.function.arguments)
                            output = func(**args)
                            output = self.validate_tool_output(output, func_name)
                            tool_outputs.append({
                                "tool_call_id": tool_call.id,
                                "role": "tool",
                                "name": func_name,
                                "content": output
                            })
                            self.conversation_history.append(tool_outputs[-1])
                            print(f"  - {func_name} executed: {output[:100]}...")
                            logger.info(f"Tool {func_name} executed with args: {args}")
                        except Exception as e:
                            error_msg = json.dumps({"error": f"Tool {func_name} failed: {str(e)}"})
                            self.metrics["errors"].append(f"Tool {func_name} failed: {str(e)}")
                            logger.error(f"Tool {func_name} failed: {str(e)}")
                            self.conversation_history.append({
                                "tool_call_id": tool_call.id,
                                "role": "tool",
                                "name": func_name,
                                "content": error_msg
                            })

            # Update context with tool outputs
            self.update_context(tool_outputs, user_prompt)

            # Execute required tools if not already executed
            required_tools = [
                ("get_flight_status", {"flight_number": self.context.get("flight_number", "AA123")}),
                ("get_aviation_weather", {"icao_code": self.context.get("departure_icao", "KJFK")}),
                ("check_crew_availability", {"flight_number": self.context.get("flight_number", "AA123")}),
                ("estimate_passenger_impact", {
                    "flight_number": self.context.get("flight_number", "AA123"),
                    "delay_minutes": self.context.get("delay_minutes", 90)
                }),
                ("predict_delay_escalation", {
                    "flight_number": self.context.get("flight_number", "AA123"),
                    "delay_minutes": self.context.get("delay_minutes", 90)
                }),
                ("get_alternate_routes", {
                    "flight_number": self.context.get("flight_number", "AA123"),
                    "departure_icao": self.context.get("departure_icao", "KJFK"),
                    "destination_icao": self.context.get("destination_icao", "KLAX")
                })
            ]

            executed_tools = [t["name"] for t in tool_outputs]
            for tool_name, args in required_tools:
                if tool_name not in executed_tools:
                    result = self.execute_tool(tool_name, args)
                    if result:
                        tool_outputs.append(result)
                        self.conversation_history.append(result)
                        self.metrics["tools_called"] += 1
                        print(f"  - {tool_name} executed (manual): {result['content'][:100]}...")

            # Always execute suggest_operations_decision if not already
            if "suggest_operations_decision" not in executed_tools:
                try:
                    flight_data = self.validate_tool_output(
                        get_flight_status(self.context.get("flight_number", "AA123")),
                        "get_flight_status"
                    )
                    weather_data = self.validate_tool_output(
                        get_aviation_weather(self.context.get("departure_icao", "KJFK")),
                        "get_aviation_weather"
                    )
                    crew_data = self.validate_tool_output(
                        check_crew_availability(self.context.get("flight_number", "AA123")),
                        "check_crew_availability"
                    )
                    route_data = self.validate_tool_output(
                        get_alternate_routes(
                            self.context.get("flight_number", "AA123"),
                            self.context.get("departure_icao", "KJFK"),
                            self.context.get("destination_icao", "KLAX")
                        ),
                        "get_alternate_routes"
                    )
                    passenger_data = self.validate_tool_output(
                        estimate_passenger_impact(
                            self.context.get("flight_number", "AA123"),
                            self.context.get("delay_minutes", 90)
                        ),
                        "estimate_passenger_impact"
                    )
                    prediction_data = self.validate_tool_output(
                        predict_delay_escalation(
                            self.context.get("flight_number", "AA123"),
                            self.context.get("delay_minutes", 90)
                        ),
                        "predict_delay_escalation"
                    )
                    compliance_data = self.validate_tool_output(
                        check_regulatory_compliance(flight_data, crew_data),
                        "check_regulatory_compliance"
                    )
                    feedback_data = self.validate_tool_output(
                        estimate_passenger_feedback(
                            self.context.get("flight_number", "AA123"),
                            self.context.get("delay_minutes", 90)
                        ),
                        "estimate_passenger_feedback"
                    )
                    atc_data = self.validate_tool_output(
                        get_atc_update(self.context.get("flight_number", "AA123")),
                        "get_atc_update"
                    )
                    threshold_data = self.validate_tool_output(
                        calculate_cost_threshold(flight_data, passenger_data, prediction_data),
                        "calculate_cost_threshold"
                    )

                    decision_output = suggest_operations_decision(
                        flight_data, weather_data, crew_data, route_data, passenger_data,
                        prediction_data, compliance_data, feedback_data, atc_data, threshold_data,
                        priority
                    )

                    tool_outputs.append({
                        "tool_call_id": "manual_suggest_operations_decision",
                        "role": "tool",
                        "name": "suggest_operations_decision",
                        "content": decision_output
                    })
                    self.conversation_history.append(tool_outputs[-1])
                    self.metrics["tools_called"] += 1
                    print(f"  - suggest_operations_decision executed (manual)")
                    logger.info("Manually executed suggest_operations_decision")
                except Exception as e:
                    self.metrics["errors"].append(f"Manual suggest_operations_decision failed: {str(e)}")
                    logger.error(f"Manual suggest_operations_decision failed: {str(e)}")

            print("AOC Agent: Reflecting & synthesizing...")
            try:
                reflect_content = self.reflect(tool_outputs, priority)
                self.conversation_history.append({"role": "assistant", "content": reflect_content})
                if "Action Plan" in reflect_content:
                    final_content = reflect_content
                    logger.info("Synthesis completed with recommendation")
                    break
            except Exception as e:
                self.metrics["errors"].append(f"Reflection failed: {str(e)}")
                logger.error(f"Reflection failed: {str(e)}")
                self.conversation_history.append({
                    "role": "assistant",
                    "content": f"Reflection error: {str(e)}. Attempting fallback."
                })

        # Calculate metrics
        self.metrics["response_time"] = time.time() - start_time

        # Add metrics to final content
        metrics_summary = (
            f"\n\n### Performance Metrics\n"
            f"* Tools Called: {self.metrics['tools_called']}\n"
            f"* Response Time: {self.metrics['response_time']:.2f} seconds\n"
            f"* Errors: {len(self.metrics['errors'])}\n"
            f"* Priority: {self.context.get('priority', 'passenger')}\n"
            f"* Iterations: {iteration + 1}"
        )
        final_content += metrics_summary

        logger.info(f"Query completed in {self.metrics['response_time']:.2f} seconds")
        return final_content

# --- 6. Legacy Decision Function ---

def suggest_operations_decision_legacy(
    flight_data: str, weather_data: str, crew_data: str, route_data: str,
    passenger_data: str, prediction_data: str, compliance_data: str,
    feedback_data: str, atc_data: str, threshold_data: str, priority: str
) -> str:
    """Legacy rule-based decision function."""
    logger.info(f"Using legacy decision with priority: {priority}")

    try:
        flight = safe_json_loads(flight_data)
        weather = safe_json_loads(weather_data)
        crew = safe_json_loads(crew_data)
        routes = safe_json_loads(route_data)
        passenger = safe_json_loads(passenger_data)
        prediction = safe_json_loads(prediction_data)
        compliance = safe_json_loads(compliance_data)
        feedback = safe_json_loads(feedback_data)
        atc = safe_json_loads(atc_data)
        threshold = safe_json_loads(threshold_data)

        delay_min = flight.get("delay_minutes", 0)

        # Early return for no delay
        if delay_min == 0:
            return json.dumps({
                "recommendation": [{"action": "Proceed on schedule", "priority": 1, "risk": "low"}],
                "estimated_cost": "$0",
                "risk_level": "low",
                "type": "legacy"
            })

        # Build recommendations based on rules
        recommendation = []
        total_cost = 0

        # Determine risk level
        escalation_likelihood = prediction.get("escalation_likelihood", 0.5)
        risk_level = "high" if escalation_likelihood > 0.6 else "medium" if escalation_likelihood > 0.3 else "low"

        # Check compliance
        if compliance.get("status") != "Fully Compliant":
            recommendation.append({
                "action": "Address compliance issues immediately",
                "priority": 0,
                "risk": "high",
                "cost": "$5000"
            })

        # Priority-based decision
        if priority == "cost" and delay_min >= 120 and routes.get("alternates"):
            # Cost priority - look for cost-effective reroute
            best_route = min(routes["alternates"], key=lambda x: x.get("extra_cost", 0))
            recommendation.append({
                "action": f"Reroute via {best_route['route']} (+{best_route['extra_time_min']} min)",
                "priority": 1,
                "risk": "medium",
                "cost": f"${best_route['extra_cost']}"
            })
            recommendation.append({
                "action": f"Notify passengers of route change",
                "priority": 2,
                "risk": "low",
                "cost": "$1000"
            })
            total_cost = best_route["extra_cost"] + 1000 + passenger.get("total_cost", 0)

        elif priority == "passenger" and delay_min > 0:
            # Passenger priority - focus on amenities and communication
            amenities_cost = "$3150" if delay_min > 90 else "$900"
            recommendation.append({
                "action": f"Delay {delay_min} min with passenger amenities",
                "priority": 1,
                "risk": risk_level,
                "cost": amenities_cost
            })
            recommendation.append({
                "action": f"Notify passengers: Delay {delay_min} min",
                "priority": 2,
                "risk": "low",
                "cost": "$100"
            })
            recommendation.append({
                "action": "Monitor ATC and provide regular updates",
                "priority": 3,
                "risk": "low",
                "cost": "$0"
            })
            total_cost = (3150 if delay_min > 90 else 900) + 100

        elif priority == "safety":
            # Safety priority - conservative approach
            recommendation.append({
                "action": f"Delay {delay_min} min and conduct safety review",
                "priority": 1,
                "risk": risk_level,
                "cost": "$500"
            })
            recommendation.append({
                "action": "Consult with maintenance for any concerns",
                "priority": 2,
                "risk": "low",
                "cost": "$0"
            })
            total_cost = 500 + passenger.get("total_cost", 0)

        else:
            # Balanced approach
            recommendation.append({
                "action": f"Delay {delay_min} min with standard procedures",
                "priority": 1,
                "risk": risk_level,
                "cost": "$0"
            })
            recommendation.append({
                "action": "Monitor situation and update as needed",
                "priority": 2,
                "risk": "low",
                "cost": "$0"
            })
            total_cost = passenger.get("total_cost", 0)

        return json.dumps({
            "recommendation": recommendation,
            "estimated_cost": f"${total_cost}",
            "risk_level": risk_level,
            "type": "legacy",
            "delay_minutes": delay_min
        })

    except Exception as e:
        logger.error(f"Error in legacy decision: {str(e)}")
        return json.dumps({
            "error": f"Legacy decision failed: {str(e)}",
            "type": "legacy_error"
        })

# --- 7. Unified Decision Function ---

def suggest_operations_decision(
    flight_data: str, weather_data: str, crew_data: str, route_data: str,
    passenger_data: str, prediction_data: str, compliance_data: str,
    feedback_data: str, atc_data: str, threshold_data: str, priority: str,
    use_ai: bool = True, model: str = DEFAULT_MODEL
) -> str:
    """
    Unified decision function - uses AI or legacy based on parameter.
    """
    logger.info(f"Decision requested - AI: {use_ai}, Model: {model}, Priority: {priority}")

    if use_ai:
        return suggest_operations_decision_ai(
            flight_data, weather_data, crew_data, route_data,
            passenger_data, prediction_data, compliance_data,
            feedback_data, atc_data, threshold_data, priority,
            model=model
        )
    else:
        return suggest_operations_decision_legacy(
            flight_data, weather_data, crew_data, route_data,
            passenger_data, prediction_data, compliance_data,
            feedback_data, atc_data, threshold_data, priority
        )

# --- 8. AI Decision Function ---

def build_decision_context(parsed_data: Dict[str, Any]) -> str:
    """Build a comprehensive context string for the AI model."""
    flight = parsed_data.get('flight', {})
    weather = parsed_data.get('weather', {})
    crew = parsed_data.get('crew', {})
    routes = parsed_data.get('routes', {})
    passenger = parsed_data.get('passenger', {})
    prediction = parsed_data.get('prediction', {})
    compliance = parsed_data.get('compliance', {})
    feedback = parsed_data.get('feedback', {})
    atc = parsed_data.get('atc', {})
    threshold = parsed_data.get('threshold', {})

    delay_min = flight.get('delay_minutes', 0)
    flight_number = flight.get('flight_number', 'Unknown')

    context = f"""
    FLIGHT OPERATIONS CONTEXT
    =========================

    FLIGHT INFORMATION:
    • Flight Number: {flight_number}
    • Status: {flight.get('status', 'unknown').upper()}
    • Current Delay: {delay_min} minutes
    • Gate: {flight.get('gate', 'Unknown')}
    • Aircraft: {flight.get('aircraft', 'Unknown')}
    • Route: {flight.get('departure_icao', 'Unknown')} → {flight.get('destination_icao', 'Unknown')}
    • ETA: {flight.get('eta', 'Unknown')}

    WEATHER CONDITIONS:
    • Airport: {weather.get('icao', 'Unknown')}
    • Conditions: {weather.get('conditions', 'Unknown')}
    • Temperature: {weather.get('temperature', 'N/A')}°C
    • Wind Speed: {weather.get('wind_speed', 'N/A')} knots
    • Visibility: {weather.get('visibility', 'N/A')} miles
    • METAR: {weather.get('metar', 'Unknown')}

    CREW STATUS:
    • Available: {'YES' if crew.get('available', False) else 'NO'}
    • Pilot: {crew.get('pilot', 'Unknown')}
    • Copilot: {crew.get('copilot', 'Unknown')}
    • Total Crew: {crew.get('total_crew', 0)}
    • Rest Compliance: {'YES' if crew.get('rest_compliance', False) else 'NO'}

    PASSENGER IMPACT:
    • Passengers: {passenger.get('passengers', 0)}
    • Rebooking Cost: ${passenger.get('rebooking_cost', 0):,}
    • Meal Vouchers: ${passenger.get('meal_vouchers', 0):,}
    • Total Passenger Cost: ${passenger.get('total_cost', 0):,}

    PREDICTIONS:
    • Escalation Likelihood: {prediction.get('escalation_likelihood', 0) * 100:.1f}%
    • Predicted Additional Delay: {prediction.get('predicted_additional_delay', 0)} minutes
    • Prediction Confidence: {prediction.get('confidence', 0) * 100:.1f}%

    REGULATORY COMPLIANCE:
    • Status: {compliance.get('status', 'Unknown')}
    • Compliance Score: {compliance.get('compliance_score', 0) * 100:.1f}%
    • Issues: {', '.join(compliance.get('issues', [])) or 'None'}
    • Warnings: {', '.join(compliance.get('warnings', [])) or 'None'}

    PASSENGER FEEDBACK:
    • Satisfaction Score: {feedback.get('satisfaction_score', 0):.2f}
    • Sentiment: {feedback.get('feedback', 'Unknown')}
    • Delay Impact: {feedback.get('delay_impact', 'Unknown')}

    ATC UPDATE:
    • Status: {atc.get('status', 'Unknown')}
    • Additional Delay: {atc.get('additional_delay_min', 0)} minutes
    • Confidence: {atc.get('confidence', 'Unknown')}

    COST ANALYSIS:
    • Delay Cost (Current): ${threshold.get('delay_cost', 0):,}
    • Delay Cost with Escalation: ${threshold.get('delay_cost_with_escalation', 0):,}
    • Cancellation Cost: ${threshold.get('cancel_cost', 0):,}
    • Cost Difference: ${threshold.get('cost_difference', 0):,}
    • Cost Recommendation: {threshold.get('recommendation', 'Unknown')}

    ROUTE ALTERNATIVES:
    • Original: {routes.get('original_route', 'Unknown')}
    • Alternatives: {len(routes.get('alternates', []))} available
    """

    # Add route details
    for i, alt in enumerate(routes.get('alternates', []), 1):
        context += f"\n    {i}. {alt.get('route', 'Unknown')} (+{alt.get('extra_time_min', 0)} min, ${alt.get('extra_cost', 0):,})"

    return context

def extract_json_from_response(response: str) -> Optional[Dict]:
    """Extract JSON from AI response with multiple strategies."""
    # Strategy 1: Try to parse the entire response as JSON
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        pass

    # Strategy 2: Look for JSON in markdown code blocks
    code_block_pattern = r'```(?:json)?\s*(\{.*?\})\s*```'
    matches = re.findall(code_block_pattern, response, re.DOTALL)
    for match in matches:
        try:
            return json.loads(match)
        except json.JSONDecodeError:
            continue

    # Strategy 3: Find JSON-like structures with regex
    json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    matches = re.findall(json_pattern, response, re.DOTALL)

    matches.sort(key=len, reverse=True)
    for match in matches:
        try:
            cleaned = match.strip()
            cleaned = re.sub(r',\s*}', '}', cleaned)
            cleaned = re.sub(r',\s*]', ']', cleaned)
            return json.loads(cleaned)
        except json.JSONDecodeError:
            continue

    return None

def suggest_operations_decision_ai(
    flight_data: str, weather_data: str, crew_data: str, route_data: str,
    passenger_data: str, prediction_data: str, compliance_data: str,
    feedback_data: str, atc_data: str, threshold_data: str, priority: str,
    model: str = DEFAULT_MODEL) -> str:
    """Use GLM-5.2 AI model to generate intelligent operations decisions."""
    logger.info(f"Generating AI-powered decision with {model}, priority: {priority}")

    try:
        # Parse all input data
        parsed_data = {}
        data_map = {
            'flight': flight_data,
            'weather': weather_data,
            'crew': crew_data,
            'routes': route_data,
            'passenger': passenger_data,
            'prediction': prediction_data,
            'compliance': compliance_data,
            'feedback': feedback_data,
            'atc': atc_data,
            'threshold': threshold_data
        }

        for name, data_str in data_map.items():
            parsed = safe_json_loads(data_str, {})
            parsed_data[name] = parsed

        # Build context
        context = build_decision_context(parsed_data)

        flight_number = parsed_data.get('flight', {}).get('flight_number', 'Unknown')
        delay_min = parsed_data.get('flight', {}).get('delay_minutes', 0)

        # System and user prompts
        system_prompt = """You are a senior aviation operations manager with over 20 years of experience."""

        user_prompt = f"""
        Based on the flight operations data provided, generate a decision recommendation prioritizing {priority.upper()}.

        DATA:
        {context}

        Provide your recommendation as a JSON object:
        {{
            "primary_recommendation": "Clear action to take",
            "secondary_actions": ["Action 1", "Action 2", "Action 3"],
            "risk_assessment": {{
                "level": "Low|Medium-Low|Medium|Medium-High|High",
                "rationale": "Why this risk level"
            }},
            "estimated_cost": "Specific cost estimate",
            "passenger_impact": {{
                "current_impact": "Current passenger situation",
                "expected_impact": "Expected after actions",
                "mitigation": "How to help passengers"
            }},
            "timeline": {{
                "immediate_actions": "0-30 min",
                "short_term": "30-120 min",
                "medium_term": "2-6 hours"
            }},
            "justification": "Detailed reasoning"
        }}

        Return ONLY valid JSON.
        """

        # Call API
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.2,
                max_tokens=2000,
                top_p=0.85
            )
            ai_response = completion.choices[0].message.content
        except Exception as api_error:
            logger.error(f"API call failed: {str(api_error)}")
            raise

        # Extract JSON
        decision_data = extract_json_from_response(ai_response)

        if not decision_data:
            decision_data = {
                "primary_recommendation": "AI analysis completed but JSON parsing failed",
                "secondary_actions": ["Please check the full response for details"],
                "risk_assessment": {"level": "Medium", "rationale": "Unable to parse full response"},
                "estimated_cost": "$0",
                "passenger_impact": {
                    "current_impact": "See full response",
                    "expected_impact": "See full response",
                    "mitigation": "See full response"
                },
                "timeline": {
                    "immediate_actions": "See full response",
                    "short_term": "See full response",
                    "medium_term": "See full response"
                },
                "justification": ai_response[:500] + ("..." if len(ai_response) > 500 else ""),
                "raw_response": ai_response
            }

        # Ensure required fields exist
        required_fields = ['primary_recommendation', 'secondary_actions', 'risk_assessment',
                         'estimated_cost', 'passenger_impact', 'timeline', 'justification']

        for field in required_fields:
            if field not in decision_data:
                if field == 'secondary_actions':
                    decision_data[field] = []
                elif field == 'risk_assessment':
                    decision_data[field] = {"level": "Medium", "rationale": "Not specified"}
                elif field == 'passenger_impact':
                    decision_data[field] = {"current_impact": "Not specified", "expected_impact": "Not specified", "mitigation": "Not specified"}
                elif field == 'timeline':
                    decision_data[field] = {"immediate_actions": "Not specified", "short_term": "Not specified", "medium_term": "Not specified"}
                else:
                    decision_data[field] = "Not specified"

        # Add metadata
        decision_data['model_used'] = model
        decision_data['priority'] = priority
        decision_data['flight_number'] = flight_number
        decision_data['timestamp'] = datetime.now().isoformat()
        decision_data['delay_minutes'] = delay_min

        return json.dumps(decision_data, indent=2)

    except Exception as e:
        logger.error(f"Critical error in AI decision making: {str(e)}", exc_info=True)
        return json.dumps({
            "error": f"AI decision failed: {str(e)}",
            "primary_recommendation": "Proceed with standard delay procedures",
            "secondary_actions": ["Monitor situation", "Communicate with passengers", "Coordinate with ATC"],
            "risk_assessment": {"level": "Medium", "rationale": "Fallback decision due to AI error"},
            "estimated_cost": "$0",
            "passenger_impact": {"current_impact": "Under evaluation", "expected_impact": "To be determined", "mitigation": "To be determined"},
            "timeline": {"immediate_actions": "Begin standard procedures", "short_term": "Monitor and update", "medium_term": "Re-evaluate"},
            "justification": "Generated as fallback due to AI processing error",
            "timestamp": datetime.now().isoformat(),
            "priority": priority,
            "fallback": True
        }, indent=2)

# --- 9. Main Execution ---

if __name__ == "__main__":
    agent = AOCCAgent()

    # Test query with passenger priority
    prompt = "Flight AA123 is delayed by 45 minutes, prioritize passenger satisfaction. Provide a prioritized action plan with risks and costs."
    print("=" * 70)
    print("QUERY 1: Passenger Priority")
    print("=" * 70)
    result = agent.run_query(prompt)
    print(result)

    # Test query with cost priority
    follow_up = "Now, what if the delay worsens to 120 minutes? Update the plan with alternate routes, notify passengers, and prioritize cost."
    print("\n" + "=" * 70)
    print("QUERY 2: Cost Priority with 120-min Delay")
    print("=" * 70)
    result = agent.run_query(follow_up)
    print(result)

QUERY 1: Passenger Priority
AOC Agent: Iteration 1 - Planning...
AOC Agent: Executing tools...
  - get_flight_status executed: {"flight_number": "AA123", "status": "delayed", "delay_minutes": 45, "eta": "2025-10-01T18:30:00Z", ...
  - check_crew_availability executed: {"available": true, "pilot": "Available (ID: P001, Rest: 10h)", "copilot": "Available (ID: C001, Res...
  - estimate_passenger_impact executed: {"flight_number": "AA123", "passengers": 158, "rebooking_cost": 0, "compensation_cost": 0, "meal_vou...
  - predict_delay_escalation executed: {"escalation_likelihood": 0.423, "predicted_additional_delay": 32, "time_factor": 1.3, "confidence":...
  - estimate_passenger_feedback executed: {"flight_number": "AA123", "satisfaction_score": 0.642, "feedback": "Neutral", "sentiment_score": 0....
  - get_atc_update executed: {"status": "Holding pattern expected", "additional_delay_min": 45, "confidence": "Medium", "flight_n...
  - get_aviation_weather executed (manual): {"icao": "KJFK", 

## PART 3: THINKING MODE DEMONSTRATION

In [7]:
# ============================================================
# PART 3: THINKING MODE DEMONSTRATION - FINAL COMPLETE VERSION
# ============================================================
# This section demonstrates Chain-of-Thought reasoning with GLM-5.2
# showing how the model thinks through complex aviation decisions
# before providing final recommendations.
#
# FIXES INCLUDED:
# 1. Complete recommendation extraction from thinking
# 2. Full sentence capture (no truncation)
# 3. Multiple extraction strategies
# 4. Robust fallback handling
# 5. Clean, production-ready code

import json
import time
import re
from datetime import datetime
from typing import Dict, Any, Optional, List
import logging

logger = logging.getLogger(__name__)

# --- 3.1 Configuration ---

class ThinkingConfig:
    """Configuration for thinking mode."""
    ENABLED = True
    SHOW_REASONING = True
    MAX_TOKENS = 3500
    TEMPERATURE = 0.3
    TOP_P = 0.9
    MODEL = "glm-5.2"

    SECTION_HEADERS = [
        "STEP 1: SITUATION ANALYSIS",
        "STEP 2: DATA ASSESSMENT",
        "STEP 3: RISK EVALUATION",
        "STEP 4: OPTIONS ANALYSIS",
        "STEP 5: TRADE-OFF ASSESSMENT",
        "STEP 6: DECISION RATIONALE"
    ]

# --- 3.2 Complete Recommendation Extraction ---

def extract_full_recommendation(thinking: str, full_response: str = "") -> str:
    """
    Extract the COMPLETE primary recommendation from thinking text.
    Uses multiple strategies to find the full, coherent recommendation.
    """

    # Strategy 1: Look for explicit final recommendation section
    if full_response:
        patterns = [
            r'FINAL RECOMMENDATION\s*[:：]\s*([^.]*(?:[.][^.]*){0,10})',
            r'RECOMMENDATION\s*[:：]\s*([^.]*(?:[.][^.]*){0,10})',
            r'DECISION\s*[:：]\s*([^.]*(?:[.][^.]*){0,10})',
            r'---\s*FINAL\s*RECOMMENDATION\s*---\s*([^.]*(?:[.][^.]*){0,10})',
            r'```\s*([^.]*(?:[.][^.]*){0,10})\s*```',
        ]
        for pattern in patterns:
            match = re.search(pattern, full_response, re.DOTALL | re.IGNORECASE)
            if match:
                recommendation = match.group(1).strip()
                recommendation = clean_recommendation(recommendation)
                if len(recommendation) > 30:
                    return recommendation

    # Strategy 2: Look for conclusive sentences in the thinking
    patterns = [
        r'(?:I|we|the\s+recommendation|the\s+decision)\s+(?:recommend|propose|suggest|decide|choose|select|opt\s+for)\s+([^.!]+[.!])',
        r'(?:therefore|thus|consequently|as\s+a\s+result|in\s+conclusion|ultimately|finally|overall)\s*(?:,|:|\s+)([^.!]+[.!])',
        r'(?:best|optimal|preferred|recommended)\s+(?:course|action|option|approach)\s+(?:is|would\s+be|should\s+be)\s+([^.!]+[.!])',
        r'(?:we|i)\s+(?:should|will|must|need\s+to|ought\s+to)\s+([^.!]+[.!])',
        r'(?:decision|action|plan|approach)\s+(?:is|will\s+be|should\s+be|recommended)\s+([^.!]+[.!])',
        r'(?:our|my)\s+(?:recommendation|decision|conclusion|final\s+thought)\s+(?:is|:)\s+([^.!]+[.!])',
    ]

    for pattern in patterns:
        matches = re.findall(pattern, thinking, re.IGNORECASE)
        if matches:
            for match in reversed(matches):
                recommendation = match.strip()
                recommendation = clean_recommendation(recommendation)
                if len(recommendation) > 20:
                    return recommendation

    # Strategy 3: Look for "Option X" in the final section
    option_pattern = r'(?:option|scenario|alternative)\s*([A-Za-z0-9])\s*(?:is|:|\s+selected|\s+chosen|\s+recommended)\s+([^.!]+[.!])'
    matches = re.findall(option_pattern, thinking, re.IGNORECASE)
    if matches:
        option, description = matches[-1]
        recommendation = f"Option {option}: {description.strip()}"
        return clean_recommendation(recommendation)

    # Strategy 4: Find the last 3 sentences that contain action words
    sentences = re.split(r'[.!?]\s+', thinking)
    action_words = ['delay', 'reroute', 'cancel', 'hold', 'proceed', 'communicate', 'notify',
                   'provide', 'monitor', 'coordinate', 'dispatch', 'substitute', 'divert']

    candidates = []
    for sentence in reversed(sentences):
        sentence = sentence.strip()
        if len(sentence) > 30:
            # Check if sentence contains any action word
            if any(word in sentence.lower() for word in action_words):
                candidates.append(sentence)
                if len(candidates) >= 2:
                    break

    if candidates:
        # Combine the candidates into a coherent recommendation
        recommendation = " ".join(reversed(candidates))
        if len(recommendation) > 20:
            return clean_recommendation(recommendation)

    # Strategy 5: Look for the last substantive paragraph
    paragraphs = re.split(r'\n\s*\n', thinking)
    if paragraphs:
        for para in reversed(paragraphs):
            para = para.strip()
            if len(para) > 50 and any(word in para.lower() for word in ['recommend', 'propose', 'suggest', 'decision', 'action']):
                # Take the first sentence of the paragraph
                first_sentence = re.split(r'[.!?]\s+', para)[0]
                if len(first_sentence) > 30:
                    return clean_recommendation(first_sentence)

    # Strategy 6: Look for any sentence with "I recommend" or "we recommend"
    for sentence in sentences:
        if re.search(r'(?:i|we)\s+recommend', sentence, re.IGNORECASE):
            return clean_recommendation(sentence.strip())

    # Final fallback
    return "See detailed thinking for complete recommendation"

def clean_recommendation(text: str) -> str:
    """Clean up a recommendation string."""
    # Remove extra whitespace
    text = ' '.join(text.split())

    # Remove leading/trailing punctuation
    text = re.sub(r'^[,\s:;]+', '', text)
    text = re.sub(r'[,\s:;]+$', '', text)

    # Ensure it ends with a period
    if text and text[-1] not in '.!?':
        text += '.'

    # Capitalize first letter
    if text:
        text = text[0].upper() + text[1:] if len(text) > 1 else text.upper()

    return text

# --- 3.3 Complete JSON Extraction ---

def extract_decision_from_thinking(full_response: str) -> Dict:
    """
    Complete decision extraction from thinking output.
    Captures ALL fields including full recommendation.
    """
    decision = {}

    # Strategy 1: Try to find JSON
    json_patterns = [
        r'```json\s*(\{[\s\S]*?\})\s*```',
        r'```\s*(\{[\s\S]*?\})\s*```',
        r'\{[\s\S]*"primary_recommendation"[\s\S]*\}',
        r'\{[\s\S]*"recommendation"[\s\S]*\}',
    ]

    for pattern in json_patterns:
        match = re.search(pattern, full_response, re.DOTALL)
        if match:
            json_str = match.group(1) if match.groups() else match.group(0)
            try:
                cleaned = clean_json_string(json_str)
                parsed = json.loads(cleaned)
                if parsed and isinstance(parsed, dict):
                    decision = parsed
                    break
            except:
                continue

    # Strategy 2: If JSON found but has placeholder recommendation, extract from thinking
    if decision:
        primary = decision.get('primary_recommendation', '')
        if not primary or primary.startswith('See thinking') or primary == 'See thinking output for recommendation':
            # Extract from thinking
            thinking = extract_thinking_section(full_response)
            if thinking:
                full_recommendation = extract_full_recommendation(thinking, full_response)
                if full_recommendation and not full_recommendation.startswith('See'):
                    decision['primary_recommendation'] = full_recommendation

                    # Also try to extract other fields from thinking
                    if 'estimated_cost' not in decision or decision['estimated_cost'] == 'See reasoning':
                        cost = extract_cost_from_thinking(thinking)
                        if cost:
                            decision['estimated_cost'] = cost

                    if 'justification' not in decision or decision['justification'] == 'Full justification in thinking above':
                        justification = extract_justification_from_thinking(thinking)
                        if justification:
                            decision['justification'] = justification

    # Strategy 3: If no JSON found, extract everything from thinking
    if not decision:
        thinking = extract_thinking_section(full_response)
        if thinking:
            decision = {
                'primary_recommendation': extract_full_recommendation(thinking, full_response),
                'secondary_actions': extract_actions_from_thinking(thinking),
                'risk_assessment': extract_risk_from_thinking(thinking),
                'estimated_cost': extract_cost_from_thinking(thinking),
                'passenger_impact': extract_passenger_impact_from_thinking(thinking),
                'timeline': extract_timeline_from_thinking(thinking),
                'justification': extract_justification_from_thinking(thinking)
            }

    # Strategy 4: Final fallback
    if not decision or not decision.get('primary_recommendation'):
        decision = {
            'primary_recommendation': extract_full_recommendation(full_response, full_response),
            'secondary_actions': ['See detailed reasoning for actions'],
            'risk_assessment': {'level': 'Medium', 'rationale': 'See reasoning'},
            'estimated_cost': 'See reasoning for cost breakdown',
            'passenger_impact': {'current': 'See reasoning', 'expected': 'See reasoning', 'mitigation': 'See reasoning'},
            'timeline': {'immediate': 'See reasoning', 'short_term': 'See reasoning', 'medium_term': 'See reasoning'},
            'justification': 'See detailed thinking for complete justification'
        }

    return decision

def extract_thinking_section(full_response: str) -> str:
    """Extract the thinking section from the full response."""
    patterns = [
        r'(?:STEP \d+:.*?)(?=FINAL RECOMMENDATION|---\s*$|\Z)',
        r'(?:THINKING|REASONING)\s*[:：]\s*([\s\S]*?)(?:FINAL RECOMMENDATION|---\s*$)',
        r'([\s\S]*?)(?=```json|\{[\s\S]*"primary_recommendation")',
    ]

    for pattern in patterns:
        match = re.search(pattern, full_response, re.DOTALL | re.IGNORECASE)
        if match:
            thinking = match.group(1) if match.groups() else match.group(0)
            thinking = thinking.strip()
            if len(thinking) > 100:
                return thinking

    # Fallback: take everything before the last JSON
    json_match = re.search(r'\{[\s\S]*\}', full_response)
    if json_match:
        thinking = full_response[:json_match.start()].strip()
        if len(thinking) > 100:
            return thinking

    return full_response[:1500]

def extract_actions_from_thinking(thinking: str) -> List[str]:
    """Extract secondary actions from thinking."""
    actions = []

    patterns = [
        r'(?:secondary|additional|recommended)\s+actions?[:\s]+([^.]*(?:[.][^.]*){0,5})',
        r'(?:actions?|steps)\s+to\s+take[:\s]+([^.]*(?:[.][^.]*){0,5})',
        r'(?:next\s+steps|follow-up|implementation)[:\s]+([^.]*(?:[.][^.]*){0,5})',
    ]

    for pattern in patterns:
        match = re.search(pattern, thinking, re.IGNORECASE)
        if match:
            text = match.group(1).strip()
            # Split into individual actions
            items = re.split(r'[,;]\s*|\.\s+', text)
            for item in items:
                item = item.strip()
                if len(item) > 15 and not item.startswith('See'):
                    actions.append(item)
            if actions:
                break

    # If no actions found, look for numbered lists
    if not actions:
        numbered = re.findall(r'\d+\.\s*([^.!]+[.!])', thinking)
        if numbered:
            actions = [item.strip() for item in numbered[:5]]

    # If still no actions, look for bullet points
    if not actions:
        bullets = re.findall(r'[•\-*]\s*([^.!]+[.!])', thinking)
        if bullets:
            actions = [item.strip() for item in bullets[:5]]

    return actions[:5] if actions else ['See detailed reasoning for actions']

def extract_cost_from_thinking(thinking: str) -> str:
    """Extract cost from thinking."""
    patterns = [
        r'(?:estimated\s+)?cost\s*(?:is|:|\s+of)\s*\$?([\d,]+(?:\s*-\s*[\d,]+)?)',
        r'\$\s*([\d,]+(?:\s*-\s*[\d,]+)?)',
        r'(?:total|overall)\s+cost\s*(?:is|:|\s+of)\s*\$?([\d,]+)',
        r'cost\s+estimate\s*[:：]\s*\$?([\d,]+)',
    ]

    for pattern in patterns:
        match = re.search(pattern, thinking, re.IGNORECASE)
        if match:
            cost = match.group(1).strip()
            if not cost.startswith('$'):
                cost = f"${cost}"
            return cost

    return 'See reasoning for cost breakdown'

def extract_risk_from_thinking(thinking: str) -> Dict:
    """Extract risk assessment from thinking."""
    risk = {'level': 'Medium', 'rationale': 'See reasoning'}

    # Extract risk level
    risk_patterns = [
        r'risk\s+(?:level|assessment)\s*(?:is|:)\s*([A-Za-z-]+)',
        r'(High|Medium|Low)\s+risk',
        r'risk\s*[:：]\s*([A-Za-z-]+)',
    ]

    for pattern in risk_patterns:
        match = re.search(pattern, thinking, re.IGNORECASE)
        if match:
            level = match.group(1).strip().capitalize()
            if level in ['High', 'Medium', 'Low', 'Medium-Low', 'Medium-High']:
                risk['level'] = level
                break

    # Extract rationale
    rationale_patterns = [
        r'risk\s+(?:rationale|reason|justification)[:\s]+([^.!]+[.!])',
        r'risk\s+assessment[:\s]+([^.!]+[.!])',
        r'risk\s+(?:is|:)\s+[A-Za-z-]+\s+because\s+([^.!]+[.!])',
    ]

    for pattern in rationale_patterns:
        match = re.search(pattern, thinking, re.IGNORECASE)
        if match:
            rationale = match.group(1).strip()
            if len(rationale) > 10:
                risk['rationale'] = rationale
                break

    return risk

def extract_passenger_impact_from_thinking(thinking: str) -> Dict:
    """Extract passenger impact from thinking."""
    impact = {
        'current': 'See reasoning',
        'expected': 'See reasoning',
        'mitigation': 'See reasoning'
    }

    # Look for passenger impact sections
    section_match = re.search(
        r'passenger\s+impact[:\s]+([^.]*(?:[.][^.]*){0,5})',
        thinking, re.IGNORECASE
    )
    if section_match:
        impact['current'] = section_match.group(1).strip()

    # Look for mitigation
    mitigation_match = re.search(
        r'(?:mitigation|compensation|accommodation|rebooking)[:\s]+([^.]*(?:[.][^.]*){0,5})',
        thinking, re.IGNORECASE
    )
    if mitigation_match:
        impact['mitigation'] = mitigation_match.group(1).strip()

    return impact

def extract_timeline_from_thinking(thinking: str) -> Dict:
    """Extract timeline from thinking."""
    timeline = {
        'immediate': 'See reasoning',
        'short_term': 'See reasoning',
        'medium_term': 'See reasoning'
    }

    # Look for timeline sections
    timeline_match = re.search(
        r'timeline[:\s]+([^.]*(?:[.][^.]*){0,5})',
        thinking, re.IGNORECASE
    )
    if timeline_match:
        timeline_text = timeline_match.group(1).strip()
        timeline['immediate'] = timeline_text[:100]
        timeline['short_term'] = timeline_text[100:200] if len(timeline_text) > 100 else timeline_text

    # Look for specific timeframes
    immediate_match = re.search(r'(?:0-30|immediate)[:\s]+([^.!]+[.!])', thinking, re.IGNORECASE)
    if immediate_match:
        timeline['immediate'] = immediate_match.group(1).strip()

    short_match = re.search(r'(?:30-120|short\s+term)[:\s]+([^.!]+[.!])', thinking, re.IGNORECASE)
    if short_match:
        timeline['short_term'] = short_match.group(1).strip()

    medium_match = re.search(r'(?:2-6|medium\s+term)[:\s]+([^.!]+[.!])', thinking, re.IGNORECASE)
    if medium_match:
        timeline['medium_term'] = medium_match.group(1).strip()

    return timeline

def extract_justification_from_thinking(thinking: str) -> str:
    """Extract justification from thinking."""
    justification_patterns = [
        r'justification[:\s]+([^.]*(?:[.][^.]*){0,10})',
        r'rationale[:\s]+([^.]*(?:[.][^.]*){0,10})',
        r'(?:because|since|as\s+the|this\s+is\s+because)[:\s]+([^.]*(?:[.][^.]*){0,10})',
        r'(?:reason|why)[:\s]+([^.]*(?:[.][^.]*){0,10})',
    ]

    for pattern in justification_patterns:
        match = re.search(pattern, thinking, re.IGNORECASE)
        if match:
            justification = match.group(1).strip()
            if len(justification) > 30:
                return justification[:300]

    # Look for the conclusion paragraph
    paragraphs = re.split(r'\n\s*\n', thinking)
    for para in reversed(paragraphs):
        if 'therefore' in para.lower() or 'conclusion' in para.lower() or 'recommend' in para.lower():
            sentences = re.split(r'[.!?]\s+', para)
            if sentences:
                combined = ' '.join(sentences[:3])
                if len(combined) > 50:
                    return combined[:300]

    return 'Full justification provided in detailed thinking above'

def clean_json_string(json_str: str) -> str:
    """Clean common JSON formatting issues."""
    json_str = re.sub(r',\s*}', '}', json_str)
    json_str = re.sub(r',\s*]', ']', json_str)
    json_str = re.sub(r'([{,])\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*:', r'\1"\2":', json_str)
    json_str = re.sub(r"'", '"', json_str)
    json_str = re.sub(r'//.*?$', '', json_str, flags=re.MULTILINE)
    json_str = re.sub(r'/\*.*?\*/', '', json_str, flags=re.DOTALL)
    return json_str

# --- 3.4 Thinking Mode Core Function ---

def think_through_problem(
    context_data: Dict[str, Any],
    priority: str,
    client: OpenAI,
    model: str = "glm-5.2"
) -> Dict[str, Any]:
    """
    Generate thinking/reasoning process using GLM-5.2.
    Forces structured step-by-step reasoning before final decision.
    Uses complete extraction to capture all decision fields.
    """
    logger.info(f"🧠 THINKING MODE: Reasoning with priority: {priority}")

    context = build_thinking_context(context_data)

    # Strong, explicit prompt with format requirements
    reasoning_prompt = f"""
**CRITICAL INSTRUCTION: You MUST show your complete step-by-step reasoning process BEFORE giving any final recommendation.**

You are a senior aviation operations manager with 20+ years of experience. Analyze this situation thoroughly and systematically.

{context}

**PRIORITY:** {priority.upper()}

**REQUIRED OUTPUT FORMAT:**

You MUST follow this exact structure:

---
STEP 1: SITUATION ANALYSIS
- What is the current flight status?
- What are the key operational parameters?
- What is the context of this situation?
- What are the immediate concerns?

STEP 2: DATA ASSESSMENT
- Flight status: [analyze]
- Weather conditions: [analyze]
- Crew status: [analyze]
- Passenger impact: [analyze]
- Predictions: [analyze]
- ATC status: [analyze]
- Alternate routes: [analyze]
- Cost analysis: [analyze]
- Compliance status: [analyze]

STEP 3: RISK EVALUATION
- List all identified risks with severity (High/Medium/Low)
- Explain the likelihood of each risk
- Identify which risks are most critical

STEP 4: OPTIONS ANALYSIS
- Option A: [describe action, cost, timeline, pros, cons]
- Option B: [describe action, cost, timeline, pros, cons]
- Option C: [describe action, cost, timeline, pros, cons]
- (Include at least 3 viable options)

STEP 5: TRADE-OFF ASSESSMENT
- Compare each option against the priority: {priority.upper()}
- Analyze pros and cons of each option
- Identify the optimal balance

STEP 6: DECISION RATIONALE
- Which option is recommended and WHY
- How it addresses the priority
- Why other options were rejected
- Confidence level in this decision

---
**FINAL RECOMMENDATION:**

Provide your final decision as a JSON object with these exact fields:

{{
    "primary_recommendation": "Clear, complete action statement describing exactly what to do",
    "secondary_actions": ["Action 1", "Action 2", "Action 3"],
    "risk_assessment": {{
        "level": "Low/Medium/High",
        "rationale": "Why this risk level"
    }},
    "estimated_cost": "Specific cost estimate with breakdown",
    "passenger_impact": {{
        "current_impact": "Current passenger situation",
        "expected_impact": "Expected after actions",
        "mitigation": "How to help passengers"
    }},
    "timeline": {{
        "immediate_actions": "0-30 minutes",
        "short_term": "30-120 minutes",
        "medium_term": "2-6 hours"
    }},
    "justification": "Detailed reasoning summary"
}}

**IMPORTANT:**
1. The JSON must be valid and properly formatted
2. Use double quotes for all strings and keys
3. The primary_recommendation MUST be a complete, coherent sentence
4. Include specific details (flight numbers, gates, etc.)
---
"""

    print("\n" + "="*70)
    print("🧠 THINKING MODE: Generating structured reasoning...")
    print("="*70)
    print("⏳ Model is analyzing the problem step by step...\n")

    start_time = time.time()

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": """You are a senior aviation operations manager.
                    You ALWAYS show your complete step-by-step reasoning before providing any recommendation.
                    You are thorough, analytical, and structured in your thinking.
                    You never skip steps or provide final answers without showing your work.
                    Your final recommendation MUST be a valid JSON object.
                    The primary_recommendation MUST be a complete, coherent sentence with specific details."""
                },
                {"role": "user", "content": reasoning_prompt}
            ],
            temperature=ThinkingConfig.TEMPERATURE,
            max_tokens=ThinkingConfig.MAX_TOKENS,
            top_p=ThinkingConfig.TOP_P
        )

        full_response = response.choices[0].message.content
        total_time = time.time() - start_time

        # Extract thinking section
        thinking = extract_thinking_section(full_response)

        # ===== COMPLETE DECISION EXTRACTION =====
        decision = extract_decision_from_thinking(full_response)

        # Verify the recommendation was properly extracted
        if decision.get('primary_recommendation', '').startswith('See thinking'):
            # Try one more time with a more aggressive extraction
            thinking = extract_thinking_section(full_response)
            if thinking:
                full_recommendation = extract_full_recommendation(thinking, full_response)
                if full_recommendation and not full_recommendation.startswith('See'):
                    decision['primary_recommendation'] = full_recommendation

        # Count thinking depth
        thinking_words = len(thinking.split())
        has_structured = any(
            header.lower() in thinking.lower()
            for header in ThinkingConfig.SECTION_HEADERS
        )

        print(f"⏱️ Response generated in {total_time:.2f} seconds")
        print(f"📝 Thinking depth: {thinking_words} words")
        print(f"📊 Structured reasoning: {'✅ Yes' if has_structured else '⚠️ Partial'}")
        print(f"📋 Decision extracted: {'✅ Yes' if decision else '❌ No'}")

        # Display thinking process
        print("\n" + "─"*70)
        print("💭 THINKING PROCESS:")
        print("─"*70)

        thinking_display = thinking[:3000] + ("..." if len(thinking) > 3000 else "")
        print(thinking_display)
        print("─"*70)

        # Display decision
        print("\n" + "="*70)
        print("🎯 FINAL DECISION:")
        print("="*70)
        print(json.dumps(decision, indent=2))

        result = {
            "thinking": thinking,
            "thinking_words": thinking_words,
            "has_structured": has_structured,
            "decision": decision,
            "total_time": total_time,
            "priority": priority,
            "timestamp": datetime.now().isoformat(),
            "model_used": model
        }

        return result

    except Exception as e:
        logger.error(f"Thinking mode failed: {str(e)}")
        return {
            "error": str(e),
            "thinking": f"Error: {str(e)}",
            "decision": {"error": str(e), "fallback": True},
            "total_time": time.time() - start_time
        }

# --- 3.5 Build Context Function ---

def build_thinking_context(data: Dict[str, Any]) -> str:
    """Build detailed context for thinking mode with clear sections."""
    flight = data.get('flight', {})
    weather = data.get('weather', {})
    crew = data.get('crew', {})
    passenger = data.get('passenger', {})
    prediction = data.get('prediction', {})
    atc = data.get('atc', {})
    routes = data.get('routes', {})
    compliance = data.get('compliance', {})
    threshold = data.get('threshold', {})

    return f"""
FLIGHT OPERATIONS DATA
======================

FLIGHT INFORMATION:
- Flight Number: {flight.get('flight_number', 'Unknown')}
- Status: {flight.get('status', 'unknown').upper()}
- Current Delay: {flight.get('delay_minutes', 0)} minutes
- Gate: {flight.get('gate', 'Unknown')}
- Aircraft: {flight.get('aircraft', 'Unknown')}
- Route: {flight.get('departure_icao', 'Unknown')} → {flight.get('destination_icao', 'Unknown')}
- ETA: {flight.get('eta', 'Unknown')}

WEATHER CONDITIONS:
- Airport: {weather.get('icao', 'Unknown')}
- Conditions: {weather.get('conditions', 'Unknown')}
- Temperature: {weather.get('temperature', 'N/A')}°C
- Wind Speed: {weather.get('wind_speed', 'N/A')} knots
- Visibility: {weather.get('visibility', 'N/A')} miles
- METAR: {weather.get('metar', 'Unknown')}

CREW STATUS:
- Available: {'YES' if crew.get('available', False) else 'NO'}
- Pilot: {crew.get('pilot', 'Unknown')}
- Copilot: {crew.get('copilot', 'Unknown')}
- Cabin Crew: {len(crew.get('cabin_crew', []))} crew members
- Total Crew: {crew.get('total_crew', 0)}
- Rest Compliance: {'YES' if crew.get('rest_compliance', False) else 'NO'}

PASSENGER IMPACT:
- Passengers Affected: {passenger.get('passengers', 0)}
- Rebooking Cost: ${passenger.get('rebooking_cost', 0):,}
- Compensation Cost: ${passenger.get('compensation_cost', 0):,}
- Meal Vouchers: ${passenger.get('meal_vouchers', 0):,}
- Hotel Cost: ${passenger.get('hotel_cost', 0):,}
- Total Passenger Cost: ${passenger.get('total_cost', 0):,}

PREDICTIONS:
- Escalation Likelihood: {prediction.get('escalation_likelihood', 0) * 100:.1f}%
- Predicted Additional Delay: {prediction.get('predicted_additional_delay', 0)} minutes
- Time Factor: {prediction.get('time_factor', 1.0)}
- Prediction Confidence: {prediction.get('confidence', 0) * 100:.1f}%

ATC UPDATE:
- Status: {atc.get('status', 'No update')}
- Additional Delay: {atc.get('additional_delay_min', 0)} minutes
- Confidence: {atc.get('confidence', 'Unknown')}
- Timestamp: {atc.get('timestamp', 'Unknown')}

ALTERNATE ROUTES:
- Original: {routes.get('original_route', 'Unknown')}
- Alternatives Available: {len(routes.get('alternates', []))}
{format_routes(routes.get('alternates', []))}

REGULATORY COMPLIANCE:
- Status: {compliance.get('status', 'Unknown')}
- Compliance Score: {compliance.get('compliance_score', 0) * 100:.1f}%
- Issues: {', '.join(compliance.get('issues', [])) or 'None'}
- Warnings: {', '.join(compliance.get('warnings', [])) or 'None'}

COST ANALYSIS:
- Current Delay Cost: ${threshold.get('delay_cost', 0):,}
- Delay Cost with Escalation: ${threshold.get('delay_cost_with_escalation', 0):,}
- Cancellation Cost: ${threshold.get('cancel_cost', 0):,}
- Cost Difference: ${threshold.get('cost_difference', 0):,}
- Threshold: {threshold.get('threshold_min', 120)} minutes
- Cost Recommendation: {threshold.get('recommendation', 'Unknown')}
- Confidence: {threshold.get('confidence', 0) * 100:.1f}%
"""

def format_routes(routes: List[Dict]) -> str:
    """Format alternate routes for display."""
    if not routes:
        return "- No alternate routes available"

    result = ""
    for i, route in enumerate(routes, 1):
        result += f"""
  {i}. {route.get('route', 'Unknown')}
     - Extra Time: +{route.get('extra_time_min', 0)} minutes
     - Extra Cost: ${route.get('extra_cost', 0):,}
     - Extra Fuel: {route.get('extra_fuel', 0)} lbs
     - Risk: {route.get('risk', 'unknown')}
    """
    return result

# --- 3.6 Comparison Function ---

def compare_standard_vs_thinking(
    context_data: Dict[str, Any],
    priority: str,
    client: OpenAI
) -> Dict[str, Any]:
    """Compare standard mode vs thinking mode performance."""
    print("\n" + "="*70)
    print("📊 COMPARISON: Standard Mode vs Thinking Mode")
    print("="*70)

    results = {}

    # 1. Run Standard Mode (PART 1 style)
    print("\n🤖 Running STANDARD MODE (no thinking)...")
    standard_start = time.time()

    try:
        flight_data = json.dumps(context_data.get('flight', {}))
        weather_data = json.dumps(context_data.get('weather', {}))
        crew_data = json.dumps(context_data.get('crew', {}))
        route_data = json.dumps(context_data.get('routes', {}))
        passenger_data = json.dumps(context_data.get('passenger', {}))
        prediction_data = json.dumps(context_data.get('prediction', {}))
        compliance_data = json.dumps(context_data.get('compliance', {}))
        feedback_data = json.dumps(context_data.get('feedback', {}))
        atc_data = json.dumps(context_data.get('atc', {}))
        threshold_data = json.dumps(context_data.get('threshold', {}))

        standard_result = suggest_operations_decision_ai(
            flight_data=flight_data,
            weather_data=weather_data,
            crew_data=crew_data,
            route_data=route_data,
            passenger_data=passenger_data,
            prediction_data=prediction_data,
            compliance_data=compliance_data,
            feedback_data=feedback_data,
            atc_data=atc_data,
            threshold_data=threshold_data,
            priority=priority
        )
        standard_time = time.time() - standard_start
        standard_decision = safe_json_loads(standard_result, {})

        results['standard'] = {
            'decision': standard_decision,
            'time': standard_time,
            'has_thinking': False
        }
        print(f"   ✅ Completed in {standard_time:.2f}s")

    except Exception as e:
        print(f"   ❌ Failed: {str(e)}")
        results['standard'] = {'error': str(e)}

    # 2. Run Thinking Mode
    print("\n🧠 Running THINKING MODE...")
    thinking_result = think_through_problem(context_data, priority, client)
    results['thinking'] = thinking_result
    print(f"   ✅ Completed in {thinking_result.get('total_time', 0):.2f}s")

    # 3. Compare
    print("\n" + "="*70)
    print("📊 COMPARISON RESULTS")
    print("="*70)

    if 'decision' in results.get('standard', {}) and 'decision' in results.get('thinking', {}):
        std = results['standard']['decision']
        think = results['thinking']['decision']

        thinking_primary = think.get('primary_recommendation', 'N/A')
        thinking_extracted = not thinking_primary.startswith('See thinking')

        print(f"""
┌──────────────────────────────────────────────────────────────────┐
│                    DECISION COMPARISON                          │
├──────────────────────────────────────────────────────────────────┤
│ PRIMARY ACTION:                                                 │
│   Standard: {std.get('primary_recommendation', 'N/A')[:70]}...   │
│   Thinking: {think.get('primary_recommendation', 'N/A')[:70]}...   │
├──────────────────────────────────────────────────────────────────┤
│ RISK LEVEL:                                                     │
│   Standard: {std.get('risk_assessment', {}).get('level', 'N/A')}                    │
│   Thinking: {think.get('risk_assessment', {}).get('level', 'N/A')}                    │
├──────────────────────────────────────────────────────────────────┤
│ ESTIMATED COST:                                                 │
│   Standard: {std.get('estimated_cost', 'N/A')}                   │
│   Thinking: {think.get('estimated_cost', 'N/A')}                   │
├──────────────────────────────────────────────────────────────────┤
│ RESPONSE TIME:                                                  │
│   Standard: {results['standard']['time']:.2f}s                  │
│   Thinking: {results['thinking']['total_time']:.2f}s            │
├──────────────────────────────────────────────────────────────────┤
│ THINKING DEPTH:                                                 │
│   Standard: No reasoning shown                                  │
│   Thinking: {results['thinking'].get('thinking_words', 0)} words    │
│   Structured: {'✅ Yes' if results['thinking'].get('has_structured') else '❌ No'}    │
│   Extracted: {'✅ Yes' if thinking_extracted else '⚠️ Partial'}    │
├──────────────────────────────────────────────────────────────────┤
│ VERDICT:                                                        │
│   Thinking mode provides:                                       │
│   • Step-by-step reasoning                                      │
│   • Transparent decision-making                                 │
│   • Detailed justification                                      │
│   • Audit trail for compliance                                  │
│   • Better for complex, high-stakes decisions                   │
└──────────────────────────────────────────────────────────────────┘
        """)

    return results

# --- 3.7 Multi-Scenario Testing ---

def test_thinking_mode_scenarios(client: OpenAI):
    """Test thinking mode across multiple scenarios."""
    print("\n" + "="*70)
    print("🧠 THINKING MODE: Multi-Scenario Testing")
    print("="*70)

    scenarios = [
        {"name": "Passenger Priority", "priority": "passenger", "flight": "AA123", "delay": 45},
        {"name": "Cost Priority", "priority": "cost", "flight": "UA456", "delay": 120},
        {"name": "Safety Priority", "priority": "safety", "flight": "DL789", "delay": 180},
        {"name": "Balanced Priority", "priority": "balanced", "flight": "SWA123", "delay": 90}
    ]

    results = []

    for i, scenario in enumerate(scenarios, 1):
        print(f"\n{'='*70}")
        print(f"📋 SCENARIO {i}: {scenario['name']}")
        print(f"   Flight: {scenario['flight']} | Delay: {scenario['delay']} min | Priority: {scenario['priority'].upper()}")
        print('='*70)

        # Gather data
        flight_data = get_flight_status(scenario['flight'])
        flight = safe_json_loads(flight_data)
        flight['delay_minutes'] = scenario['delay']
        flight_data = json.dumps(flight)

        departure = flight.get('departure_icao', 'KJFK')
        destination = flight.get('destination_icao', 'KLAX')

        weather_data = get_aviation_weather(departure)
        crew_data = check_crew_availability(scenario['flight'])
        passenger_data = estimate_passenger_impact(scenario['flight'], scenario['delay'])
        prediction_data = predict_delay_escalation(scenario['flight'], scenario['delay'])
        atc_data = get_atc_update(scenario['flight'])
        route_data = get_alternate_routes(scenario['flight'], departure, destination)
        compliance_data = check_regulatory_compliance(flight_data, crew_data)
        feedback_data = estimate_passenger_feedback(scenario['flight'], scenario['delay'])
        threshold_data = calculate_cost_threshold(flight_data, passenger_data, prediction_data)

        context_data = {
            'flight': safe_json_loads(flight_data),
            'weather': safe_json_loads(weather_data),
            'crew': safe_json_loads(crew_data),
            'passenger': safe_json_loads(passenger_data),
            'prediction': safe_json_loads(prediction_data),
            'atc': safe_json_loads(atc_data),
            'routes': safe_json_loads(route_data),
            'compliance': safe_json_loads(compliance_data),
            'feedback': safe_json_loads(feedback_data),
            'threshold': safe_json_loads(threshold_data)
        }

        # Run thinking mode
        result = think_through_problem(context_data, scenario['priority'], client)
        results.append({
            'scenario': scenario,
            'result': result
        })

        # Quick summary
        if result.get('decision'):
            decision = result['decision']
            primary = decision.get('primary_recommendation', 'N/A')

            print(f"\n📌 QUICK SUMMARY:")
            print(f"   Primary: {primary[:120]}..." if len(primary) > 120 else f"   Primary: {primary}")
            print(f"   Risk: {decision.get('risk_assessment', {}).get('level', 'N/A')}")
            print(f"   Cost: {decision.get('estimated_cost', 'N/A')}")

    # Summary
    print("\n" + "="*70)
    print("📊 MULTI-SCENARIO SUMMARY")
    print("="*70)

    for i, r in enumerate(results, 1):
        s = r['scenario']
        res = r['result']
        decision = res.get('decision', {})
        primary = decision.get('primary_recommendation', 'N/A')

        print(f"""
Scenario {i}: {s['name']}
├─ Flight: {s['flight']} (Delay: {s['delay']} min)
├─ Priority: {s['priority']}
├─ Thinking Depth: {res.get('thinking_words', 0)} words
├─ Structured: {'✅' if res.get('has_structured') else '❌'}
├─ Time: {res.get('total_time', 0):.2f}s
└─ Decision: {primary[:80]}...
        """)

    return results

# --- 3.8 Main Demo Function ---

def run_thinking_mode_demo():
    """Main demonstration of thinking mode with complete extraction."""
    print("\n" + "="*70)
    print("🧠 GLM-5.2 THINKING MODE DEMONSTRATION")
    print("📋 COMPLETE RECOMMENDATION EXTRACTION")
    print("="*70)
    print("\nThis demonstrates how GLM-5.2 performs step-by-step reasoning")
    print("before making aviation operations decisions.")
    print("\nKey Features:")
    print("  ✓ Forced structured reasoning (6 steps)")
    print("  ✓ Transparent decision-making")
    print("  ✓ Trade-off analysis")
    print("  ✓ Risk assessment")
    print("  ✓ Comparison with standard mode")
    print("  ✓ COMPLETE recommendation extraction (no truncation)")

    # Validate API
    if not ZAI_KEY_API:
        print("\n❌ Error: ZAI_KEY not configured")
        return

    client = OpenAI(
        api_key=ZAI_KEY_API,
        base_url="https://api.z.ai/api/paas/v4/"
    )

    # Test scenario
    flight_number = "AA123"
    priority = "passenger"
    delay_min = 45

    print(f"\n📋 TEST SCENARIO:")
    print(f"   Flight: {flight_number}")
    print(f"   Delay: {delay_min} minutes")
    print(f"   Priority: {priority.upper()}")

    # Gather data
    print("\n⏳ Gathering flight data...")
    flight_data = get_flight_status(flight_number)
    flight = safe_json_loads(flight_data)
    flight['delay_minutes'] = delay_min
    flight_data = json.dumps(flight)

    departure = flight.get('departure_icao', 'KJFK')
    destination = flight.get('destination_icao', 'KLAX')

    weather_data = get_aviation_weather(departure)
    crew_data = check_crew_availability(flight_number)
    passenger_data = estimate_passenger_impact(flight_number, delay_min)
    prediction_data = predict_delay_escalation(flight_number, delay_min)
    atc_data = get_atc_update(flight_number)
    route_data = get_alternate_routes(flight_number, departure, destination)
    compliance_data = check_regulatory_compliance(flight_data, crew_data)
    feedback_data = estimate_passenger_feedback(flight_number, delay_min)
    threshold_data = calculate_cost_threshold(flight_data, passenger_data, prediction_data)

    context_data = {
        'flight': safe_json_loads(flight_data),
        'weather': safe_json_loads(weather_data),
        'crew': safe_json_loads(crew_data),
        'passenger': safe_json_loads(passenger_data),
        'prediction': safe_json_loads(prediction_data),
        'atc': safe_json_loads(atc_data),
        'routes': safe_json_loads(route_data),
        'compliance': safe_json_loads(compliance_data),
        'feedback': safe_json_loads(feedback_data),
        'threshold': safe_json_loads(threshold_data)
    }

    print("✅ Data gathered successfully!")

    # Run comparison
    compare_standard_vs_thinking(context_data, priority, client)

    # Run multi-scenario
    test_thinking_mode_scenarios(client)

    # Final summary
    print("\n" + "="*70)
    print("✅ THINKING MODE DEMO COMPLETE")
    print("="*70)
    print("\n💡 KEY TAKEAWAYS:")
    print("  1. Thinking mode shows the model's step-by-step reasoning")
    print("  2. Provides transparency and auditability")
    print("  3. Essential for high-stakes decisions")
    print("  4. Takes ~2x time but produces more reasoned outputs")
    print("  5. Enables trust in AI decision-making")
    print("  6. COMPLETE recommendation extraction now working")
    print("\n🔧 Implementation Notes:")
    print("  • Use explicit step-by-step prompts (STEP 1:, STEP 2:, etc.)")
    print("  • Set temperature=0.3 for consistent formatting")
    print("  • Use multiple strategies for recommendation extraction")
    print("  • Parse thinking separately from final decision")
    print("  • Display thinking for transparency")
    print("  • Use for complex, multi-factor decisions")
    print("="*70)

# --- 3.9 Run the Demo ---

if __name__ == "__main__":
    run_thinking_mode_demo()


🧠 GLM-5.2 THINKING MODE DEMONSTRATION
📋 COMPLETE RECOMMENDATION EXTRACTION

This demonstrates how GLM-5.2 performs step-by-step reasoning
before making aviation operations decisions.

Key Features:
  ✓ Forced structured reasoning (6 steps)
  ✓ Transparent decision-making
  ✓ Trade-off analysis
  ✓ Risk assessment
  ✓ Comparison with standard mode
  ✓ COMPLETE recommendation extraction (no truncation)

📋 TEST SCENARIO:
   Flight: AA123
   Delay: 45 minutes
   Priority: PASSENGER

⏳ Gathering flight data...
✅ Data gathered successfully!

📊 COMPARISON: Standard Mode vs Thinking Mode

🤖 Running STANDARD MODE (no thinking)...
   ✅ Completed in 25.23s

🧠 Running THINKING MODE...

🧠 THINKING MODE: Generating structured reasoning...
⏳ Model is analyzing the problem step by step...

⏱️ Response generated in 48.62 seconds
📝 Thinking depth: 896 words
📊 Structured reasoning: ✅ Yes
📋 Decision extracted: ✅ Yes

──────────────────────────────────────────────────────────────────────
💭 THINKING PROCES